<a href="https://colab.research.google.com/github/nmansour67/skills-introduction-to-github/blob/main/GenAI_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# PIPELINE MASTER NOTEBOOK
# DBA Thesis: Impact of Integrating GenAI in Health Economics
# and Business Models of Academic Medical Centers in Lebanon
# ============================================================
# EXECUTION ORDER:
#   Block 0 → Data Ingestion & Simulation
#   Block 1 → Qualitative ML Analysis
#   Block 2 → Quantitative Analysis
#   Block 3 → Mixed Methods Integration
# ============================================================

# ─────────────────────────────────────────────────────────────
# CELL 0.1 — ENVIRONMENT SETUP
# ─────────────────────────────────────────────────────────────

# Persist output directory across the session
from google.colab import drive
drive.mount("/content/drive")

import os
OUTPUT_DIR = "/content/drive/MyDrive/DBA_Thesis_Outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory ready: {OUTPUT_DIR}")

import os
os.makedirs("/content/thesis_outputs/", exist_ok=True)
OUTPUT_DIR = "/content/thesis_outputs/"

import subprocess, sys

def install(packages):
    for pkg in packages:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", pkg, "-q"]
        )

install([
    "faker", "langdetect", "sentence-transformers",
    "bertopic", "umap-learn", "hdbscan",
    "factor_analyzer", "pingouin", "shap",
    "networkx", "plotly", "kaleido",
    "vaderSentiment", "spacy", "nltk",
    "scikit-learn", "pandas", "numpy",
    "matplotlib", "seaborn", "openpyxl"
])

import subprocess
subprocess.run(
    ["python", "-m", "spacy", "download", "en_core_web_sm", "-q"]
)

import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
import seaborn as sns
warnings.filterwarnings("ignore")

# Global seed
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Output directory
OUTPUT_DIR = "/content/thesis_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment ready.")
print(f"📁 Output folder: {OUTPUT_DIR}")




Mounted at /content/drive
✅ Output directory ready: /content/drive/MyDrive/DBA_Thesis_Outputs/
✅ Environment ready.
📁 Output folder: /content/thesis_outputs/


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 0.2 — PIPELINE PROGRESS DASHBOARD
# ─────────────────────────────────────────────────────────────
pipeline_log = {
    "Block 0: Data Ingestion"         : "⏳ Pending",
    "Block 1: Qualitative ML"         : "⏳ Pending",
    "Block 2: Quantitative Analysis"  : "⏳ Pending",
    "Block 3: Mixed Methods"          : "⏳ Pending",
}
findings_log   = []
innovations_log = []
files_saved    = []

def print_dashboard():
    print("\n" + "="*55)
    print("   DBA THESIS — PIPELINE DASHBOARD")
    print("="*55)
    for block, status in pipeline_log.items():
        print(f"  {status}  {block}")
    print("-"*55)
    print(f"  📌 Thesis findings flagged : {len(findings_log)}")
    print(f"  🔬 Innovations flagged     : {len(innovations_log)}")
    print(f"  💾 Files saved             : {len(files_saved)}")
    print("="*55 + "\n")

def log_finding(text):
    findings_log.append(text)
    print(f"  📌 THESIS FINDING: {text}")

def log_innovation(text):
    innovations_log.append(text)
    print(f"  🔬 INNOVATION: {text}")

def save_file(filepath, df_or_fig=None, kind="csv"):
    full_path = OUTPUT_DIR + filepath
    if kind == "csv" and df_or_fig is not None:
        df_or_fig.to_csv(full_path, index=False)
    elif kind == "fig" and df_or_fig is not None:
        df_or_fig.savefig(full_path, dpi=300,
                          bbox_inches="tight")
        plt.close()
    files_saved.append(filepath)
    print(f"  💾 Saved: {filepath}")
    return full_path

print_dashboard()





   DBA THESIS — PIPELINE DASHBOARD
  ⏳ Pending  Block 0: Data Ingestion
  ⏳ Pending  Block 1: Qualitative ML
  ⏳ Pending  Block 2: Quantitative Analysis
  ⏳ Pending  Block 3: Mixed Methods
-------------------------------------------------------
  📌 Thesis findings flagged : 0
  🔬 Innovations flagged     : 0
  💾 Files saved             : 0



In [4]:
# ─────────────────────────────────────────────────────────────
# CELL 0.3 — DATA INGESTION GATE (CLEAN FINAL VERSION)
# ─────────────────────────────────────────────────────────────

import io
import pandas as pd
from google.colab import files as colab_files

def read_file_auto(file_bytes, filename):
    if filename.endswith(".xlsx"):
        df = pd.read_excel(io.BytesIO(file_bytes))
        print("   ✅ Excel file read successfully")
        return df
    for enc in ["utf-8","utf-8-sig","latin-1","cp1256","cp1252","iso-8859-1","windows-1256"]:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=enc)
            print(f"   ✅ Encoding: {enc}")
            return df
        except Exception:
            continue
    df = pd.read_csv(io.BytesIO(file_bytes), encoding="utf-8", encoding_errors="replace")
    print("   ⚠️  Fallback encoding applied")
    return df

print("="*55)
print("   DATA INGESTION GATE")
print("="*55)
print("\n  [1] USE REAL DATA\n  [2] USE SYNTHETIC DATA\n")
data_mode = input("Enter 1 or 2: ").strip()

if data_mode == "1":

    # ── FILE A: QUALITATIVE ───────────────────────────────────
    print("\n" + "─"*55)
    print("  STEP 1 OF 2 — Upload QUALITATIVE file (File A)")
    print("─"*55)
    print("\n👇 Click Choose Files and select File A:\n")

    uploaded_a = colab_files.upload()
    filename_a = list(uploaded_a.keys())[0]
    print(f"\n✅ Received: {filename_a}")

    raw_qual = read_file_auto(uploaded_a[filename_a], filename_a)
    print(f"   Shape: {raw_qual.shape}")

    required_qual = ["participant_id","role","institution","source_type","transcript_text"]
    has_required  = all(c in raw_qual.columns for c in required_qual)

    if has_required:
        qual_df = raw_qual.copy()
        print("   ✅ Standard format — no transformation needed.")

    else:
        print("\n   ⚙️  Raw survey format detected — auto-transforming...")
        question_cols = [c for c in raw_qual.columns if c != "Timestamp"]

        raw_qual["transcript_text"] = raw_qual[question_cols].apply(
            lambda row: " | ".join([
                "Q: " + str(col) + " A: " + str(val)
                for col, val in zip(question_cols, row)
                if pd.notna(val) and str(val).strip() not in ["","nan"]
            ]), axis=1
        )

        raw_qual["participant_id"] = [
            "P" + str(i+1).zfill(3) for i in range(len(raw_qual))
        ]

        role_col = next(
            (c for c in raw_qual.columns
             if c.lower() in ["role","position","job_title","title","participant_role"]),
            None
        )

        if role_col:
            raw_qual["role"] = raw_qual[role_col].astype(str).str.strip().str.lower()
            print(f"   ✅ Role column found: {role_col}")
        else:
            print("\n   ⚠️  No role column found.")
            print("   [A] Same role for all")
            print("   [B] Equal split across 3 roles automatically")
            print("   [C] Enter each role manually")
            rc = input("\n   Enter A, B or C: ").strip().upper()

            role_map = {"1":"executive","2":"health_economist","3":"clinician_researcher"}

            if rc == "A":
                print("   1=executive  2=health_economist  3=clinician_researcher")
                r = input("   Enter 1, 2 or 3: ").strip()
                raw_qual["role"] = role_map.get(r, "executive")

            elif rc == "C":
                assigned = []
                for i in range(len(raw_qual)):
                    pid = raw_qual["participant_id"].iloc[i]
                    r = input(pid + " role [1/2/3]: ").strip()
                    assigned.append(role_map.get(r, "clinician_researcher"))
                raw_qual["role"] = assigned

            else:
                n     = len(raw_qual)
                third = n // 3
                rem   = n % 3
                raw_qual["role"] = (
                    ["executive"] * third +
                    ["health_economist"] * third +
                    ["clinician_researcher"] * (third + rem)
                )
                print(f"   ✅ Equal split: {third} / {third} / {third+rem}")

        inst_col = next(
            (c for c in raw_qual.columns
             if c.lower() in ["institution","hospital","center","organisation","organization","amc","facility"]),
            None
        )

        if inst_col:
            raw_qual["institution"] = raw_qual[inst_col]
            print(f"   ✅ Institution column: {inst_col}")
        else:
            inst = input("\n   Institution name for all (Enter = Not specified): ").strip()
            raw_qual["institution"] = inst if inst else "Not specified"

        raw_qual["source_type"] = "open_questionnaire"
        qual_df = raw_qual[required_qual].copy()
        print(f"\n   ✅ Done: {len(qual_df)} participants, {len(question_cols)} questions merged")

    qual_df["word_count"]  = qual_df["transcript_text"].apply(lambda x: len(str(x).split()))
    qual_df["arabic_flag"] = qual_df["transcript_text"].apply(
        lambda x: 1 if any('\u0600' <= c <= '\u06FF' for c in str(x)) else 0
    )

    print(f"\n  📊 Qualitative summary:")
    print(f"     Participants : {len(qual_df)}")
    print(f"     Roles        : {qual_df['role'].value_counts().to_dict()}")
    print(f"     Avg words    : {qual_df['word_count'].mean():.0f}")
    print(f"     Arabic rows  : {qual_df['arabic_flag'].sum()}")

    # ── FILE B: QUANTITATIVE ──────────────────────────────────
    print("\n" + "─"*55)
    print("  STEP 2 OF 2 — Upload QUANTITATIVE file (File B)")
    print("─"*55)
    print("\n👇 Click Choose Files and select File B:\n")

    uploaded_b = colab_files.upload()
    filename_b = list(uploaded_b.keys())[0]
    print(f"\n✅ Received: {filename_b}")

    raw_quant = read_file_auto(uploaded_b[filename_b], filename_b)
    print(f"   Shape: {raw_quant.shape}")

    q_cols_existing = [c for c in raw_quant.columns
                       if str(c).startswith("Q") and str(c)[1:].isdigit()]

    if len(q_cols_existing) >= 5:
        quant_df    = raw_quant.copy()
        likert_cols = q_cols_existing
        print("   ✅ Standard Q1...Qn format detected.")

    else:
        print("\n   ⚙️  Raw survey format — auto-renaming columns...")
        skip = ["Timestamp"]
        numeric_cols = []
        for col in raw_quant.columns:
            if col in skip:
                continue
            num_vals = pd.to_numeric(raw_quant[col], errors="coerce")
            if num_vals.notna().mean() >= 0.3:
                numeric_cols.append(col)

        rename_map  = {"Timestamp" if c == "Timestamp" else c: c
                       for c in raw_quant.columns}
        rename_map  = {col: "Q" + str(i+1) for i, col in enumerate(numeric_cols)}
        raw_quant   = raw_quant.rename(columns=rename_map)
        likert_cols = list(rename_map.values())

        pd.DataFrame({
            "item_code"     : list(rename_map.values()),
            "original_label": list(rename_map.keys())
        }).to_csv(OUTPUT_DIR + "quantitative_item_labels.csv", index=False)

        print(f"   ✅ {len(likert_cols)} items renamed Q1–Q{len(likert_cols)}")
        print("   💾 Label reference saved: quantitative_item_labels.csv")

        if "participant_id" not in raw_quant.columns:
            raw_quant["participant_id"] = [
                "P" + str(i+1).zfill(3) for i in range(len(raw_quant))
            ]

        if "role" not in raw_quant.columns:
            if len(raw_quant) == len(qual_df):
                raw_quant["role"] = qual_df["role"].values
                print("   ✅ Roles inherited from qualitative file")
            else:
                raw_quant["role"] = "executive"

        if "institution" not in raw_quant.columns:
            if len(raw_quant) == len(qual_df):
                raw_quant["institution"] = qual_df["institution"].values
            else:
                raw_quant["institution"] = "Not specified"

        quant_df = raw_quant.copy()

    for col in likert_cols:
        if col in quant_df.columns:
            quant_df[col] = pd.to_numeric(quant_df[col], errors="coerce")

    # ── Domain mapping ────────────────────────────────────────
    # ⚠️ ADJUST domain names and item ranges to match your instrument
    n_items = len(likert_cols)
    print(f"\n   Likert items detected: {n_items}")

    if n_items >= 40:
        domain_mapping = {
            "COMPETITIVE_STRATEGY"  : ["Q"+str(i) for i in range(1,6)],
            "ORGANIZATIONAL_CULTURE": ["Q"+str(i) for i in range(6,11)],
            "FINANCIAL_IMPACT"      : ["Q"+str(i) for i in range(11,16)],
            "TECHNICAL_READINESS"   : ["Q"+str(i) for i in range(16,21)],
            "CLINICAL_OUTCOMES"     : ["Q"+str(i) for i in range(21,26)],
            "ETHICS_GOVERNANCE"     : ["Q"+str(i) for i in range(26,31)],
            "EDUCATION_RESEARCH"    : ["Q"+str(i) for i in range(31,36)],
            "IMPLEMENTATION"        : ["Q"+str(i) for i in range(36,41)],
        }
    elif n_items >= 25:
        domain_mapping = {
            "GENAI_READINESS"     : ["Q"+str(i) for i in range(1,6)],
            "ECON_IMPACT"         : ["Q"+str(i) for i in range(6,11)],
            "BIZ_MODEL_TRANSFORM" : ["Q"+str(i) for i in range(11,16)],
            "ACADEMIC_ALIGNMENT"  : ["Q"+str(i) for i in range(16,21)],
            "ETHICAL_CONCERN"     : ["Q"+str(i) for i in range(21,26)],
        }
    else:
        per = max(1, n_items // 5)
        domain_mapping = {
            "DOMAIN_" + str(d+1): ["Q"+str(i) for i in range(d*per+1, min((d+1)*per+1, n_items+1))]
            for d in range(5)
        }

    print(f"   Domain mapping ({len(domain_mapping)} domains):")
    for d, items in domain_mapping.items():
        present = [i for i in items if i in quant_df.columns]
        print(f"     {d:28s}: {len(present)} items")

    missing_pct = quant_df[likert_cols].isna().mean().mean() * 100
    print(f"\n  📊 Quantitative summary:")
    print(f"     Participants : {quant_df['participant_id'].nunique()}")
    print(f"     Items        : {len(likert_cols)}")
    print(f"     Missing      : {missing_pct:.1f}%")

    # ── Cross-file validation ─────────────────────────────────
    print("\n" + "─"*55)
    print("  CROSS-FILE VALIDATION")
    print("─"*55)
    qual_ids   = set(qual_df["participant_id"].unique())
    quant_ids  = set(quant_df["participant_id"].unique())
    common     = qual_ids & quant_ids
    qual_only  = qual_ids  - quant_ids
    quant_only = quant_ids - qual_ids
    print(f"  Matched IDs    : {len(common)}")
    if qual_only:
        print(f"  ⚠️  Qual only  : {len(qual_only)}")
    if quant_only:
        print(f"  ⚠️  Quant only : {len(quant_only)}")
    if not qual_only and not quant_only:
        print("  ✅ Perfect ID match")

    qual_df.to_csv(OUTPUT_DIR + "qualitative_master.csv", index=False)
    quant_df.to_csv(OUTPUT_DIR + "quantitative_questionnaire.csv", index=False)

    DATA_SOURCE    = "REAL"
    N_PARTICIPANTS = len(qual_df)
    print(f"\n✅ FILES SAVED — {N_PARTICIPANTS} participants")
    print(f"▶  Run Cell 0.4 next")

# ═════════════════════════════════════════════════════════════
# MODE 2 — SYNTHETIC
# ═════════════════════════════════════════════════════════════
elif data_mode == "2":

    from faker import Faker
    from scipy.stats import truncnorm
    fake = Faker()

    N_PARTICIPANTS = 45
    ROLES          = ["executive","health_economist","clinician_researcher"]
    ROLE_DIST      = [15,15,15]
    INSTITUTIONS   = ["AUBMC","LAU Medical Center","Makassed General Hospital","Saint George Hospital","Hotel-Dieu de France"]
    LIKERT_SCALE   = 5

    domain_mapping = {
        "COMPETITIVE_STRATEGY"  : ["Q"+str(i) for i in range(1,6)],
        "ORGANIZATIONAL_CULTURE": ["Q"+str(i) for i in range(6,11)],
        "FINANCIAL_IMPACT"      : ["Q"+str(i) for i in range(11,16)],
        "TECHNICAL_READINESS"   : ["Q"+str(i) for i in range(16,21)],
        "CLINICAL_OUTCOMES"     : ["Q"+str(i) for i in range(21,26)],
        "ETHICS_GOVERNANCE"     : ["Q"+str(i) for i in range(26,31)],
        "EDUCATION_RESEARCH"    : ["Q"+str(i) for i in range(31,36)],
        "IMPLEMENTATION"        : ["Q"+str(i) for i in range(36,41)],
    }
    likert_cols = [q for items in domain_mapping.values() for q in items]

    participants = []
    pid = 1
    for role, n in zip(ROLES, ROLE_DIST):
        for _ in range(n):
            participants.append({
                "participant_id": "P" + str(pid).zfill(3),
                "role"          : role,
                "institution"   : random.choice(INSTITUTIONS),
            })
            pid += 1
    registry_df = pd.DataFrame(participants)

    qual_rows = []
    for _, p in registry_df.iterrows():
        text = ("Q: What strategic advantages does your center aim to achieve through GenAI? "
                "A: Our center aims to leverage GenAI for improved diagnostics and efficiency. "
                "Q: How does GenAI position your center in the regional market? "
                "A: It enhances our competitive edge through data-driven decision making.")
        qual_rows.append({
            "participant_id" : p["participant_id"],
            "role"           : p["role"],
            "institution"    : p["institution"],
            "source_type"    : "open_questionnaire",
            "transcript_text": text,
            "word_count"     : len(text.split()),
            "arabic_flag"    : 0
        })
    qual_df = pd.DataFrame(qual_rows)

    role_profiles = {
        "executive"            : [4.1,4.0,4.3,3.8,4.0,2.8,3.5,3.7],
        "health_economist"     : [3.8,3.5,4.5,3.6,3.7,3.1,3.3,3.5],
        "clinician_researcher" : [3.2,3.4,3.3,3.5,4.2,4.1,4.3,3.2],
    }

    def trunc_likert(mean, sd=0.7, n=5, lo=1, hi=5):
        a = (lo-mean)/sd
        b = (hi-mean)/sd
        vals = truncnorm.rvs(a, b, loc=mean, scale=sd, size=n)
        return np.clip(np.round(vals), lo, hi).astype(int).tolist()

    quant_rows = []
    for _, p in registry_df.iterrows():
        row = {"participant_id": p["participant_id"],
               "role": p["role"],
               "institution": p["institution"]}
        means = role_profiles[p["role"]]
        for d_idx, (domain, items) in enumerate(domain_mapping.items()):
            vals = trunc_likert(means[d_idx], n=len(items))
            for item, val in zip(items, vals):
                row[item] = val
        quant_rows.append(row)

    quant_df = pd.DataFrame(quant_rows)

    qual_df.to_csv(OUTPUT_DIR + "qualitative_master.csv", index=False)
    quant_df.to_csv(OUTPUT_DIR + "quantitative_questionnaire.csv", index=False)
    registry_df.to_csv(OUTPUT_DIR + "participant_registry.csv", index=False)

    DATA_SOURCE    = "SYNTHETIC"
    N_PARTICIPANTS = len(qual_df)
    print(f"✅ SYNTHETIC DATA — {N_PARTICIPANTS} participants generated")
    print(f"▶  Run Cell 0.4 next")

else:
    raise ValueError("❌ Invalid input. Enter 1 or 2 and re-run.")

   DATA INGESTION GATE

  [1] USE REAL DATA
  [2] USE SYNTHETIC DATA

Enter 1 or 2: 1

───────────────────────────────────────────────────────
  STEP 1 OF 2 — Upload QUALITATIVE file (File A)
───────────────────────────────────────────────────────

👇 Click Choose Files and select File A:



Saving GenAI_Healthcare_Qualitative_Raw.csv to GenAI_Healthcare_Qualitative_Raw.csv

✅ Received: GenAI_Healthcare_Qualitative_Raw.csv
   ✅ Encoding: latin-1
   Shape: (78, 41)

   ⚙️  Raw survey format detected — auto-transforming...

   ⚠️  No role column found.
   [A] Same role for all
   [B] Equal split across 3 roles automatically
   [C] Enter each role manually

   Enter A, B or C: a
   1=executive  2=health_economist  3=clinician_researcher
   Enter 1, 2 or 3: 1

   Institution name for all (Enter = Not specified): anonymous

   ✅ Done: 78 participants, 40 questions merged

  📊 Qualitative summary:
     Participants : 78
     Roles        : {'executive': 78}
     Avg words    : 420
     Arabic rows  : 0

───────────────────────────────────────────────────────
  STEP 2 OF 2 — Upload QUANTITATIVE file (File B)
───────────────────────────────────────────────────────

👇 Click Choose Files and select File B:



Saving GenAI_Healthcare_Quantitative_Raw.csv to GenAI_Healthcare_Quantitative_Raw.csv

✅ Received: GenAI_Healthcare_Quantitative_Raw.csv
   ✅ Encoding: latin-1
   Shape: (86, 41)

   ⚙️  Raw survey format — auto-renaming columns...
   ✅ 13 items renamed Q1–Q13
   💾 Label reference saved: quantitative_item_labels.csv

   Likert items detected: 13
   Domain mapping (5 domains):
     DOMAIN_1                    : 2 items
     DOMAIN_2                    : 2 items
     DOMAIN_3                    : 2 items
     DOMAIN_4                    : 2 items
     DOMAIN_5                    : 2 items

  📊 Quantitative summary:
     Participants : 86
     Items        : 13
     Missing      : 0.0%

───────────────────────────────────────────────────────
  CROSS-FILE VALIDATION
───────────────────────────────────────────────────────
  Matched IDs    : 78
  ⚠️  Quant only : 8

✅ FILES SAVED — 78 participants
▶  Run Cell 0.4 next


In [5]:
import os
import pandas as pd # Import pandas for data loading

# ─────────────────────────────────────────────────────────────
# CELL 0.4 — BLOCK 0 VALIDATION
# ─────────────────────────────────────────────────────────────
# Define OUTPUT_DIR for robustness, in case the previous cell defining it was not run
OUTPUT_DIR = "/content/thesis_outputs/"

# Load DataFrames for validation, in case the previous cell was not executed in the current session
# or if files were not successfully saved.
qual_file_path = os.path.join(OUTPUT_DIR, "qualitative_master.csv")
quant_file_path = os.path.join(OUTPUT_DIR, "quantitative_questionnaire.csv")

if not os.path.exists(qual_file_path):
    print(f"❌ ERROR: Qualitative master file not found: {qual_file_path}")
    print("Please ensure Cell 0.3 (DATA INGESTION GATE) has been executed successfully to create this file.")
    raise FileNotFoundError(f"Missing required file: {qual_file_path}")

if not os.path.exists(quant_file_path):
    print(f"❌ ERROR: Quantitative questionnaire file not found: {quant_file_path}")
    print("Please ensure Cell 0.3 (DATA INGESTION GATE) has been executed successfully to create this file.")
    raise FileNotFoundError(f"Missing required file: {quant_file_path}")

qual_df = pd.read_csv(qual_file_path)
quant_df = pd.read_csv(quant_file_path)

print("\n" + "="*55)
print("   BLOCK 0 VALIDATION")
print("="*55)

validation_results = {}

# Test 1 — File existence
required_files = [
    "qualitative_master.csv",
    "quantitative_questionnaire.csv"
]
all_exist = all(
    os.path.exists(OUTPUT_DIR + f)
    for f in required_files
)
validation_results["File Existence"] = (
    "✅ PASS" if all_exist else "❌ FAIL"
)

# Test 2 — Merge integrity
q_ids = set(qual_df["participant_id"].unique())
qt_ids = set(quant_df["participant_id"].unique())
common_ids = q_ids & qt_ids
validation_results["Merge Integrity"] = (
    f"✅ PASS — {len(common_ids)} matched IDs"
    if len(common_ids) > 0
    else "❌ FAIL — no matching IDs"
)

# Test 3 — Arabic distribution
ar_rate = qual_df["arabic_flag"].mean()
validation_results["Arabic Distribution"] = (
    f"✅ PASS — {ar_rate:.1%} arabic-flagged"
    if 0.10 <= ar_rate <= 0.35
    else f"⚠️  CHECK — {ar_rate:.1%} "
         f"(expected 15–25%)"
)

# Test 4 — Likert range
item_cols = [c for c in quant_df.columns
             if c.startswith("Q") and c[1:].isdigit()]
if item_cols:
    min_val = quant_df[item_cols].min().min()
    max_val = quant_df[item_cols].max().max()
    validation_results["Likert Range"] = (
        f"✅ PASS — range [{int(min_val)}"
        f"–{int(max_val)}]"
        if min_val >= 1 and max_val <= 7
        else f"❌ FAIL — unexpected range "
             f"[{min_val}–{max_val}]"
    )

# Test 5 — Source type coverage
sources = qual_df["source_type"].unique()
expected = {"interview", "focus_group",
            "open_questionnaire"}
validation_results["Source Coverage"] = (
    "✅ PASS — all 3 sources present"
    if expected.issubset(set(sources))
    else f"⚠️  CHECK — found: {set(sources)}"
)

for test, result in validation_results.items():
    print(f"  {result:45s}  {test}")

all_passed = all(
    "PASS" in v
    for v in validation_results.values()
)

try:
    pipeline_log["Block 0: Data Ingestion"] = (
        "✅ Complete" if all_passed else "⚠️  Complete with warnings"
    )

    print("\n" + "─"*55)
    print(f"  Data source    : {DATA_SOURCE}")
    print(f"  Participants   : {N_PARTICIPANTS}")
    print(f"  Pipeline ready : {'YES ✅' if all_passed else 'YES ⚠️  — review warnings'}")
    print("─"*55)
    print("\n▶  Run Block 1 cell next → Qualitative ML Analysis")
    print_dashboard()
except NameError:
    print("\n⚠️  Warning: pipeline_log, DATA_SOURCE, or N_PARTICIPANTS might not be defined.")
    print("Please ensure Cell 0.1 (ENVIRONMENT SETUP) and Cell 0.3 (DATA INGESTION GATE) have been run successfully.")


# ─────────────────────────────────────────────────────────────
# ▶ BLOCK 1, 2, AND 3 CELLS GO BELOW THIS LINE
# Paste each block's code cells here in sequence.
# All blocks read from OUTPUT_DIR automatically.
# No file path changes needed between blocks.
# ─────────────────────────────────────────────────────────────


   BLOCK 0 VALIDATION
  ✅ PASS                                         File Existence
  ✅ PASS — 78 matched IDs                        Merge Integrity
  ⚠️  CHECK — 0.0% (expected 15–25%)             Arabic Distribution
  ✅ PASS — range [1–5]                           Likert Range
  ⚠️  CHECK — found: {'open_questionnaire'}      Source Coverage

───────────────────────────────────────────────────────
  Data source    : REAL
  Participants   : 78
  Pipeline ready : YES ⚠️  — review warnings
───────────────────────────────────────────────────────

▶  Run Block 1 cell next → Qualitative ML Analysis

   DBA THESIS — PIPELINE DASHBOARD
  ⚠️  Complete with warnings  Block 0: Data Ingestion
  ⏳ Pending  Block 1: Qualitative ML
  ⏳ Pending  Block 2: Quantitative Analysis
  ⏳ Pending  Block 3: Mixed Methods
-------------------------------------------------------
  📌 Thesis findings flagged : 0
  🔬 Innovations flagged     : 0
  💾 Files saved             : 0



In [6]:
# ═════════════════════════════════════════════════════════════
# BLOCK 1 — QUALITATIVE ML ANALYSIS
# ═════════════════════════════════════════════════════════════

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
import seaborn as sns
warnings.filterwarnings("ignore")

OUTPUT_DIR = "/content/thesis_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Install dependencies ──────────────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])

for pkg in ["sentence-transformers","bertopic","umap-learn",
            "hdbscan","vaderSentiment","networkx","langdetect"]:
    install(pkg)

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("vader_lexicon", quiet=True)
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call(["python","-m","spacy","download","en_core_web_sm","-q"])
    nlp = spacy.load("en_core_web_sm")

print("✅ Block 1 dependencies ready")


# ─────────────────────────────────────────────────────────────
# STEP 1 — LOAD DATA
# ─────────────────────────────────────────────────────────────
qual_path = OUTPUT_DIR + "qualitative_master.csv"
qual_df   = pd.read_csv(qual_path)

print("\n── STEP 1: DATA LOADED ──")
print(f"   Shape  : {qual_df.shape}")
print(f"   Columns: {list(qual_df.columns)}")
print(f"\n   Participants by role:")
print(qual_df["role"].value_counts().to_string())
print(f"\n   Participants by source_type:")
print(qual_df["source_type"].value_counts().to_string())


# ─────────────────────────────────────────────────────────────
# STEP 2 — PREPROCESSING
# ─────────────────────────────────────────────────────────────
import re
from nltk.corpus import stopwords

stop_en = set(stopwords.words("english"))
stop_custom = {
    "hospital","said","think","know","like","just","really",
    "lebanon","lebanese","aub","aubmc","center","medical",
    "nan","q","a","also","would","could","one","may","us",
    "make","use","need","get","go","well","even","much"
}
stop_all = stop_en | stop_custom

def extract_arabic(text):
    arabic = re.findall(r'[\u0600-\u06FF\s]+', str(text))
    return " ".join(arabic).strip() if arabic else ""

def preprocess_transcript(text):
    text = str(text)
    arabic_seg = extract_arabic(text)
    arabic_flag = 1 if len(arabic_seg) > 3 else 0
    text = re.sub(r'\[AR\].*?\[/AR\]', '', text, flags=re.DOTALL)
    text = re.sub(r'[\u0600-\u06FF]', '', text)
    text = text.lower()
    text = re.sub(r'Q:.*?A:', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in stop_all and len(t) > 2]
    doc = nlp(" ".join(tokens[:500]))
    lemmas = [token.lemma_ for token in doc
              if token.lemma_ not in stop_all
              and len(token.lemma_) > 2
              and token.is_alpha]
    cleaned = " ".join(lemmas)
    return cleaned, len(lemmas), arabic_flag, arabic_seg

print("\n── STEP 2: PREPROCESSING ──")
results = qual_df["transcript_text"].apply(
    lambda x: preprocess_transcript(x)
)
qual_df["cleaned_text"]     = [r[0] for r in results]
qual_df["token_count"]      = [r[1] for r in results]
qual_df["arabic_flag"]      = [r[2] for r in results]
qual_df["arabic_segments"]  = [r[3] for r in results]

print(f"   Processed: {len(qual_df)} transcripts")
print(f"   Arabic-flagged: {qual_df['arabic_flag'].sum()}")
print(f"   Avg tokens after cleaning: {qual_df['token_count'].mean():.0f}")
print("\n   Sample (first 3 rows):")
for i in range(min(3, len(qual_df))):
    orig = str(qual_df['transcript_text'].iloc[i])[:80]
    clean = str(qual_df['cleaned_text'].iloc[i])[:80]
    print(f"   [{i+1}] BEFORE: {orig}...")
    print(f"        AFTER : {clean}...")
    print()


# ─────────────────────────────────────────────────────────────
# STEP 3 — SENTENCE EMBEDDINGS
# ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

print("── STEP 3: EMBEDDING ──")
embedder  = SentenceTransformer("all-MiniLM-L6-v2")
texts     = qual_df["cleaned_text"].fillna("").tolist()
texts     = [t if len(t.strip()) > 5 else "general healthcare" for t in texts]
embeddings = embedder.encode(texts, show_progress_bar=True)
print(f"   Embedding matrix: {embeddings.shape}")
# INNOVATION: semantic embedding preserves contextual meaning


# ─────────────────────────────────────────────────────────────
# STEP 4 — BERTOPIC MODELING
# ─────────────────────────────────────────────────────────────
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

print("\n── STEP 4: BERTOPIC ──")

n_docs = len(texts)
min_topic = max(2, min(3, n_docs // 10))

vectorizer = CountVectorizer(
    stop_words="english",
    min_df=1,
    ngram_range=(1,2)
)

topic_model = BERTopic(
    min_topic_size=min_topic,
    nr_topics="auto",
    calculate_probabilities=True,
    vectorizer_model=vectorizer,
    verbose=False
)

topics, probs = topic_model.fit_transform(texts, embeddings)
qual_df["topic_id"]          = topics
qual_df["topic_probability"] = [p.max() if hasattr(p,"max") else float(p)
                                 for p in probs]

topic_info = topic_model.get_topic_info()
print(f"   Topics found: {len(topic_info[topic_info['Topic'] != -1])}")
print(f"   Outliers (-1): {(qual_df['topic_id']==-1).sum()}")
print("\n   Top words per topic:")
for _, row in topic_info[topic_info["Topic"] != -1].head(10).iterrows():
    tid   = row["Topic"]
    words = topic_model.get_topic(tid)
    if words:
        top_w = ", ".join([w for w,_ in words[:8]])
        print(f"   Topic {tid:3d} (n={row['Count']:3d}): {top_w}")

topic_overview = pd.DataFrame({
    "topic_id" : topic_info["Topic"],
    "top_words": topic_info["Topic"].apply(
        lambda t: ", ".join([w for w,_ in topic_model.get_topic(t)[:8]])
        if topic_model.get_topic(t) else ""
    ),
    "size": topic_info["Count"]
})
topic_overview.to_csv(OUTPUT_DIR + "bertopic_topic_overview.csv", index=False)

doc_topics = qual_df[["participant_id","role","source_type",
                       "topic_id","topic_probability"]].copy()
doc_topics.to_csv(OUTPUT_DIR + "bertopic_document_topics.csv", index=False)

for tid in topic_overview["topic_id"]:
    if tid == -1:
        continue
    pct = (qual_df["topic_id"] == tid).mean() * 100
    if pct > 20:
        print(f"   THESIS FINDING: Topic {tid} covers {pct:.1f}% of documents")

print("   Saved: bertopic_topic_overview.csv")
print("   Saved: bertopic_document_topics.csv")


# ─────────────────────────────────────────────────────────────
# STEP 5 — DBA CONSTRUCT MAPPING
# ─────────────────────────────────────────────────────────────
print("\n── STEP 5: DBA CONSTRUCT MAPPING ──")

dba_constructs = {
    "GenAI Adoption & Readiness": [
        "ai","artificial","intelligence","chatgpt","genai",
        "automation","technology","digital","tool","model",
        "language","machine","learning","algorithm"
    ],
    "Health Economics Impact": [
        "cost","revenue","billing","insurance","reimbursement",
        "efficiency","budget","financial","expenditure","investment",
        "return","saving","pricing","funding","economic"
    ],
    "Business Model Transformation": [
        "business","strategy","innovation","restructure","service",
        "value","partnership","transformation","process","competitive",
        "market","positioning","workflow","operation","strategic"
    ],
    "Academic Mission Alignment": [
        "research","teaching","education","faculty","training",
        "student","academic","publication","curriculum","clinical",
        "study","trial","evidence","outcome","quality"
    ],
    "Ethical & Regulatory Concerns": [
        "ethic","privacy","regulation","compliance","risk",
        "governance","consent","bias","legal","accountability",
        "transparent","fairness","oversight","guideline","policy"
    ]
}

def compute_overlap(topic_words, construct_keywords):
    if not topic_words:
        return 0.0
    topic_set = set([w.lower() for w,_ in topic_words])
    hits = sum(1 for kw in construct_keywords
               if any(kw in tw for tw in topic_set))
    return hits / len(construct_keywords)

mapping_rows = []
for _, row in topic_overview[topic_overview["topic_id"] != -1].iterrows():
    tid   = int(row["topic_id"])
    words = topic_model.get_topic(tid)
    scores = {
        c: compute_overlap(words, kws)
        for c, kws in dba_constructs.items()
    }
    sorted_c  = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    primary   = sorted_c[0][0] if sorted_c[0][1] > 0 else "Uncategorized"
    secondary = sorted_c[1][0] if len(sorted_c) > 1 and sorted_c[1][1] > 0 else ""
    mapping_rows.append({
        "topic_id"          : tid,
        "top_words"         : row["top_words"],
        "size"              : row["size"],
        "primary_construct" : primary,
        "secondary_construct": secondary,
        **{c: round(s,3) for c,s in scores.items()}
    })

construct_map_df = pd.DataFrame(mapping_rows)
construct_map_df.to_csv(OUTPUT_DIR + "bertopic_construct_mapping.csv", index=False)
print("   Saved: bertopic_construct_mapping.csv")
print("\n   Construct assignments:")
for _, r in construct_map_df.iterrows():
    print(f"   Topic {int(r['topic_id']):3d}: {r['primary_construct']}")


# ─────────────────────────────────────────────────────────────
# STEP 6 — VISUALIZATIONS
# ─────────────────────────────────────────────────────────────
print("\n── STEP 6: VISUALIZATIONS ──")

valid_topics = qual_df[qual_df["topic_id"] != -1].copy()

if not valid_topics.empty and not construct_map_df.empty:
    valid_topics = valid_topics.merge(
        construct_map_df[["topic_id","primary_construct"]],
        on="topic_id", how="left"
    )

    # a) Topic distribution
    fig, ax = plt.subplots(figsize=(10,5))
    td = topic_overview[topic_overview["topic_id"] != -1].sort_values("size", ascending=False)
    ax.bar(td["topic_id"].astype(str), td["size"],
           color=sns.color_palette("Blues_d", len(td)))
    ax.set_xlabel("Topic ID")
    ax.set_ylabel("Document Count")
    ax.set_title("BERTopic — Topic Size Distribution")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "bertopic_topic_distribution.png", dpi=300)
    plt.close()
    print("   Saved: bertopic_topic_distribution.png")

    # b) Construct heatmap
    score_cols = list(dba_constructs.keys())
    available  = [c for c in score_cols if c in construct_map_df.columns]
    if available:
        fig, ax = plt.subplots(figsize=(10,6))
        heat_data = construct_map_df.set_index("topic_id")[available]
        sns.heatmap(heat_data, annot=True, fmt=".2f",
                    cmap="YlOrRd", ax=ax, linewidths=0.5)
        ax.set_title("Topic vs DBA Construct Overlap Scores")
        ax.set_ylabel("Topic ID")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "bertopic_construct_heatmap.png", dpi=300)
        plt.close()
        print("   Saved: bertopic_construct_heatmap.png")

    # c) Role distribution
    if "role" in valid_topics.columns and "topic_id" in valid_topics.columns:
        fig, ax = plt.subplots(figsize=(10,5))
        role_topic = pd.crosstab(valid_topics["topic_id"], valid_topics["role"])
        role_topic.plot(kind="bar", stacked=True, ax=ax,
                        colormap="Set2")
        ax.set_title("Topic Prevalence by Participant Role")
        ax.set_xlabel("Topic ID")
        ax.set_ylabel("Count")
        ax.legend(bbox_to_anchor=(1.05,1), loc="upper left")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "bertopic_role_distribution.png", dpi=300)
        plt.close()
        print("   Saved: bertopic_role_distribution.png")

    # d) Source comparison
    if "source_type" in valid_topics.columns:
        fig, ax = plt.subplots(figsize=(10,5))
        src_topic = pd.crosstab(valid_topics["topic_id"], valid_topics["source_type"])
        src_topic.plot(kind="bar", ax=ax, colormap="Paired")
        ax.set_title("Topic Distribution by Source Type")
        ax.set_xlabel("Topic ID")
        ax.set_ylabel("Count")
        ax.legend(bbox_to_anchor=(1.05,1), loc="upper left")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "bertopic_source_comparison.png", dpi=300)
        plt.close()
        print("   Saved: bertopic_source_comparison.png")
else:
    print("   ⚠️  Not enough topics for visualizations — try with more data")


# ─────────────────────────────────────────────────────────────
# STEP 7 — SENTIMENT ANALYSIS
# ─────────────────────────────────────────────────────────────
from nltk.sentiment.vader import SentimentIntensityAnalyzer

print("\n── STEP 7: SENTIMENT ANALYSIS ──")
sia = SentimentIntensityAnalyzer()

qual_df["sentiment_compound"] = qual_df["cleaned_text"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)
qual_df["sentiment_label"] = qual_df["sentiment_compound"].apply(
    lambda s: "positive" if s >= 0.05 else ("negative" if s <= -0.05 else "neutral")
)

print(f"   Sentiment distribution:")
print(qual_df["sentiment_label"].value_counts().to_string())

sent_role_df = qual_df.groupby(["role","topic_id"])["sentiment_compound"].mean().reset_index()
sent_role_df.to_csv(OUTPUT_DIR + "sentiment_by_role_topic.csv", index=False)

if not valid_topics.empty and "topic_id" in qual_df.columns:
    try:
        pivot_data = qual_df[qual_df["topic_id"] != -1].pivot_table(
            values="sentiment_compound",
            index="role",
            columns="topic_id",
            aggfunc="mean"
        )
        if not pivot_data.empty:
            fig, ax = plt.subplots(figsize=(12,4))
            sns.heatmap(pivot_data, annot=True, fmt=".2f",
                        cmap="RdYlGn", center=0, ax=ax,
                        linewidths=0.5)
            ax.set_title("Mean Sentiment by Role and Topic")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR + "sentiment_heatmap_role_topic.png", dpi=300)
            plt.close()
            print("   Saved: sentiment_heatmap_role_topic.png")

            for role in pivot_data.index:
                for tid in pivot_data.columns:
                    val = pivot_data.loc[role, tid]
                    if abs(val) > 0.3:
                        print(f"   THESIS FINDING: Role '{role}' has "
                              f"sentiment {val:.2f} on Topic {tid}")
    except Exception as e:
        print(f"   ⚠️  Sentiment heatmap skipped: {e}")

print("   Saved: sentiment_by_role_topic.csv")


# ─────────────────────────────────────────────────────────────
# STEP 8 — CO-OCCURRENCE NETWORK
# ─────────────────────────────────────────────────────────────
import networkx as nx
from itertools import combinations

print("\n── STEP 8: CO-OCCURRENCE NETWORK ──")

construct_colors = {
    "GenAI Adoption & Readiness"    : "#1A6B72",
    "Health Economics Impact"       : "#D4820A",
    "Business Model Transformation" : "#1B2A4A",
    "Academic Mission Alignment"    : "#4A7A4A",
    "Ethical & Regulatory Concerns" : "#8B0000",
    "Uncategorized"                 : "#999999"
}

keyword_to_construct = {}
for construct, keywords in dba_constructs.items():
    for kw in keywords:
        keyword_to_construct[kw] = construct

G = nx.Graph()
edge_rows = []

for _, row in qual_df.iterrows():
    tokens = str(row["cleaned_text"]).split()
    topic_keywords = []
    if row["topic_id"] != -1:
        tw = topic_model.get_topic(row["topic_id"])
        if tw:
            topic_keywords = [w for w,_ in tw[:5]]
    present = [kw for kw in topic_keywords if kw in tokens]
    for w1, w2 in combinations(present, 2):
        edge_rows.append({"word1": w1, "word2": w2})
        if G.has_edge(w1, w2):
            G[w1][w2]["weight"] += 1
        else:
            G.add_edge(w1, w2, weight=1)

if len(G.nodes) > 1:
    node_colors = []
    for node in G.nodes():
        c = keyword_to_construct.get(node, "Uncategorized")
        node_colors.append(construct_colors.get(c, "#999999"))

    fig, ax = plt.subplots(figsize=(12,10))
    pos = nx.spring_layout(G, seed=42, k=2)
    weights = [G[u][v]["weight"] for u,v in G.edges()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                           node_size=800, alpha=0.9, ax=ax)
    nx.draw_networkx_edges(G, pos, width=[w*0.5 for w in weights],
                           alpha=0.5, edge_color="#AAAAAA", ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    ax.set_title("Keyword Co-occurrence Network\n"
                 "(colored by DBA construct)")
    ax.axis("off")

    from matplotlib.patches import Patch
    legend = [Patch(color=c, label=l)
              for l,c in construct_colors.items()
              if l != "Uncategorized"]
    ax.legend(handles=legend, loc="lower left",
              fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "cooccurrence_network.png", dpi=300)
    plt.close()
    print("   Saved: cooccurrence_network.png")

    edge_df = pd.DataFrame(edge_rows)
    if not edge_df.empty:
        edge_df = edge_df.groupby(["word1","word2"]).size().reset_index(name="weight")
    edge_df.to_csv(OUTPUT_DIR + "cooccurrence_edges.csv", index=False)
    print("   Saved: cooccurrence_edges.csv")
else:
    print("   ⚠️  Not enough co-occurring keywords for network")
    pd.DataFrame(columns=["word1","word2","weight"]).to_csv(
        OUTPUT_DIR + "cooccurrence_edges.csv", index=False
    )


# ─────────────────────────────────────────────────────────────
# STEP 9 — MANUAL CODING COMPARISON TEMPLATE
# ─────────────────────────────────────────────────────────────
print("\n── STEP 9: COMPARISON TEMPLATE ──")

template_df = construct_map_df[[
    "topic_id","top_words","primary_construct","size"
]].copy()
template_df.columns = [
    "topic_id","top_words","dba_construct","bertopic_size"
]
template_df["manual_code_equivalent"] = ""
template_df["convergence_status"]     = ""
template_df.to_csv(OUTPUT_DIR + "manual_vs_ml_comparison_template.csv", index=False)
print("   Saved: manual_vs_ml_comparison_template.csv")
print("   Fill 'manual_code_equivalent' after NVivo/Atlas.ti coding.")
print("   convergence_status = Confirmed / Partial / Divergent")


# ─────────────────────────────────────────────────────────────
# STEP 10 — BLOCK 1 SUMMARY
# ─────────────────────────────────────────────────────────────
files_b1 = [
    "bertopic_topic_overview.csv",
    "bertopic_document_topics.csv",
    "bertopic_construct_mapping.csv",
    "sentiment_by_role_topic.csv",
    "manual_vs_ml_comparison_template.csv",
    "cooccurrence_edges.csv",
    "bertopic_topic_distribution.png",
    "bertopic_construct_heatmap.png",
    "bertopic_role_distribution.png",
    "bertopic_source_comparison.png",
    "sentiment_heatmap_role_topic.png",
    "cooccurrence_network.png",
]
saved = [f for f in files_b1
         if os.path.exists(OUTPUT_DIR + f)]

n_topics = len(topic_overview[topic_overview["topic_id"] != -1])
constructs_hit = construct_map_df["primary_construct"].unique().tolist() if not construct_map_df.empty else []

print("\n" + "="*55)
print("  BLOCK 1 COMPLETE — QUALITATIVE ML ANALYSIS")
print("="*55)
print(f"  Participants analyzed     : {len(qual_df)}")
print(f"  Transcripts processed     : {len(qual_df)}")
print(f"  BERTopic topics found     : {n_topics}")
print(f"  DBA constructs identified : {len(constructs_hit)}")
print(f"  Primary constructs        : {constructs_hit}")
print(f"  Arabic-flagged rows       : {qual_df['arabic_flag'].sum()}")
print(f"  Files saved               : {len(saved)} of {len(files_b1)}")
print("-"*55)
print("  FILES SAVED:")
for f in saved:
    print(f"    ✅ {f}")
missing_files = [f for f in files_b1 if f not in saved]
for f in missing_files:
    print(f"    ⚠️  {f} — not generated")
print("="*55)
print("  NEXT: Run Block 2 — Quantitative Analysis")
print("  (Cronbach Alpha → EFA → Random Forest → SHAP)")
print("="*55)

✅ Block 1 dependencies ready

── STEP 1: DATA LOADED ──
   Shape  : (78, 7)
   Columns: ['participant_id', 'role', 'institution', 'source_type', 'transcript_text', 'word_count', 'arabic_flag']

   Participants by role:
role
executive    78

   Participants by source_type:
source_type
open_questionnaire    78

── STEP 2: PREPROCESSING ──
   Processed: 78 transcripts
   Arabic-flagged: 0
   Avg tokens after cleaning: 171

   Sample (first 3 rows):
   [1] BEFORE: Q: What strategic advantages does your center aim to achieve through GenAI? A: A...
        AFTER : strategic advantage aim achieve genai nursing director university strategic aim ...

   [2] BEFORE: nan...
        AFTER : ...

   [3] BEFORE: Q: What strategic advantages does your center aim to achieve through GenAI? A: 1...
        AFTER : strategic advantage aim achieve genai enhance clinical decision support accurate...

── STEP 3: EMBEDDING ──


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

   Embedding matrix: (78, 384)

── STEP 4: BERTOPIC ──
   Topics found: 2
   Outliers (-1): 0

   Top words per topic:
   Topic   0 (n= 40): genai, advantage, strategic, aim, advantage aim, aim achieve, strategic advantage, achieve genai
   Topic   1 (n= 38): genai, patient, healthcare, improve, adoption, technology, datum, staff
   THESIS FINDING: Topic 0 covers 51.3% of documents
   THESIS FINDING: Topic 1 covers 48.7% of documents
   Saved: bertopic_topic_overview.csv
   Saved: bertopic_document_topics.csv

── STEP 5: DBA CONSTRUCT MAPPING ──
   Saved: bertopic_construct_mapping.csv

   Construct assignments:
   Topic   0: GenAI Adoption & Readiness
   Topic   1: GenAI Adoption & Readiness

── STEP 6: VISUALIZATIONS ──
   Saved: bertopic_topic_distribution.png
   Saved: bertopic_construct_heatmap.png
   Saved: bertopic_role_distribution.png
   Saved: bertopic_source_comparison.png

── STEP 7: SENTIMENT ANALYSIS ──
   Sentiment distribution:
sentiment_label
positive    68
neutral    

In [7]:
# ═════════════════════════════════════════════════════════════
# BLOCK 2 — QUANTITATIVE ANALYSIS
# ═════════════════════════════════════════════════════════════

import os, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
import seaborn as sns
from scipy import stats
warnings.filterwarnings("ignore")

OUTPUT_DIR = "/content/thesis_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Install dependencies ──────────────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])

for pkg in ["pingouin","factor_analyzer","shap","umap-learn"]:
    install(pkg)

print("✅ Block 2 dependencies ready")


# ─────────────────────────────────────────────────────────────
# STEP 1 — LOAD & VALIDATE
# ─────────────────────────────────────────────────────────────
print("\n── STEP 1: LOAD & VALIDATE ──")

quant_path = OUTPUT_DIR + "quantitative_questionnaire.csv"
quant_df   = pd.read_csv(quant_path)

print(f"   Shape  : {quant_df.shape}")
print(f"   Columns: {list(quant_df.columns)[:10]} ...")
print(f"\n   Participants by role:")
if "role" in quant_df.columns:
    print(quant_df["role"].value_counts().to_string())

# Detect Likert columns
likert_cols = [c for c in quant_df.columns
               if str(c).startswith("Q") and str(c)[1:].isdigit()]
print(f"\n   Likert items detected: {len(likert_cols)}")

# Detect scale range
all_vals = quant_df[likert_cols].values.flatten()
all_vals = all_vals[~np.isnan(all_vals.astype(float))]
scale_max = int(np.nanmax(quant_df[likert_cols].values.astype(float)))
scale_min = int(np.nanmin(quant_df[likert_cols].values.astype(float)))
print(f"   Scale range detected  : {scale_min} to {scale_max}")

# Convert to numeric
for col in likert_cols:
    quant_df[col] = pd.to_numeric(quant_df[col], errors="coerce")

# Missing values
missing_pct = quant_df[likert_cols].isna().mean() * 100
high_missing = missing_pct[missing_pct > 5]
if len(high_missing) > 0:
    print(f"\n   ⚠️  High missing (>5%):")
    print(high_missing.to_string())

# Role-stratified median imputation
if "role" in quant_df.columns:
    for col in likert_cols:
        for role in quant_df["role"].unique():
            mask = (quant_df["role"] == role) & quant_df[col].isna()
            median_val = quant_df.loc[quant_df["role"]==role, col].median()
            quant_df.loc[mask, col] = median_val
else:
    for col in likert_cols:
        quant_df[col] = quant_df[col].fillna(quant_df[col].median())

print(f"\n   Missing after imputation: "
      f"{quant_df[likert_cols].isna().sum().sum()}")

# Domain mapping — auto-scales to item count
n_items = len(likert_cols)
if n_items >= 40:
    domain_mapping = {
        "COMPETITIVE_STRATEGY"  : ["Q"+str(i) for i in range(1,6)],
        "ORGANIZATIONAL_CULTURE": ["Q"+str(i) for i in range(6,11)],
        "FINANCIAL_IMPACT"      : ["Q"+str(i) for i in range(11,16)],
        "TECHNICAL_READINESS"   : ["Q"+str(i) for i in range(16,21)],
        "CLINICAL_OUTCOMES"     : ["Q"+str(i) for i in range(21,26)],
        "ETHICS_GOVERNANCE"     : ["Q"+str(i) for i in range(26,31)],
        "EDUCATION_RESEARCH"    : ["Q"+str(i) for i in range(31,36)],
        "IMPLEMENTATION"        : ["Q"+str(i) for i in range(36,41)],
    }
elif n_items >= 25:
    domain_mapping = {
        "GENAI_READINESS"     : ["Q"+str(i) for i in range(1,6)],
        "ECON_IMPACT"         : ["Q"+str(i) for i in range(6,11)],
        "BIZ_MODEL_TRANSFORM" : ["Q"+str(i) for i in range(11,16)],
        "ACADEMIC_ALIGNMENT"  : ["Q"+str(i) for i in range(16,21)],
        "ETHICAL_CONCERN"     : ["Q"+str(i) for i in range(21,26)],
    }
else:
    per = max(1, n_items // 5)
    domain_mapping = {
        "DOMAIN_"+str(d+1): ["Q"+str(i) for i in range(d*per+1, min((d+1)*per+1, n_items+1))]
        for d in range(5)
    }
# ⚠️ ADJUST domain_mapping to match your actual instrument

domain_names = list(domain_mapping.keys())
print(f"\n   Domains ({len(domain_names)}): {domain_names}")

# Filter to items actually in dataframe
domain_mapping = {
    d: [q for q in items if q in quant_df.columns]
    for d, items in domain_mapping.items()
}
domain_mapping = {d: items for d, items in domain_mapping.items() if items}
domain_names   = list(domain_mapping.keys())


# ─────────────────────────────────────────────────────────────
# STEP 2 — DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────────
print("\n── STEP 2: DESCRIPTIVE STATISTICS ──")

desc = quant_df[likert_cols].describe().T
desc["skewness"] = quant_df[likert_cols].skew()
desc["kurtosis"] = quant_df[likert_cols].kurt()
print(desc[["mean","std","min","max","skewness","kurtosis"]].round(2).to_string())

# Build domain-item lookup
item_to_domain = {}
for d, items in domain_mapping.items():
    for q in items:
        item_to_domain[q] = d

domain_colors = dict(zip(
    domain_names,
    sns.color_palette("Set2", len(domain_names))
))

# a) Item means bar chart
fig, ax = plt.subplots(figsize=(12, max(6, len(likert_cols)*0.3)))
means = quant_df[likert_cols].mean()
sds   = quant_df[likert_cols].std()
colors_bar = [domain_colors.get(item_to_domain.get(q,""), (0.5,0.5,0.5))
              for q in likert_cols]
ax.barh(likert_cols, means, xerr=sds, color=colors_bar,
        alpha=0.8, capsize=3)
ax.set_xlabel("Mean Score")
ax.set_title("Item Means with SD by Domain")
ax.axvline(x=means.mean(), linestyle="--", color="red",
           alpha=0.5, label="Overall mean")
ax.legend()
from matplotlib.patches import Patch
legend_patches = [Patch(color=domain_colors[d], label=d)
                  for d in domain_names if d in domain_colors]
ax.legend(handles=legend_patches, loc="lower right", fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "likert_item_means.png", dpi=300)
plt.close()
print("   Saved: likert_item_means.png")

# Compute domain composites for plots
for d, items in domain_mapping.items():
    quant_df[d+"_score"] = quant_df[items].mean(axis=1)

score_cols = [d+"_score" for d in domain_names]

# b) Domain boxplots
fig, ax = plt.subplots(figsize=(12,6))
plot_data = quant_df[score_cols].copy()
plot_data.columns = domain_names
plot_data_melt = plot_data.melt(var_name="Domain", value_name="Score")
sns.boxplot(data=plot_data_melt, x="Domain", y="Score",
            palette="Set2", ax=ax, width=0.5)
sns.stripplot(data=plot_data_melt, x="Domain", y="Score",
              color="black", alpha=0.3, size=3, ax=ax, jitter=True)
ax.set_title("Domain Score Distributions")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "likert_domain_boxplots.png", dpi=300)
plt.close()
print("   Saved: likert_domain_boxplots.png")

# c) Role heatmap
if "role" in quant_df.columns:
    role_means = quant_df.groupby("role")[score_cols].mean()
    role_means.columns = domain_names
    fig, ax = plt.subplots(figsize=(10,4))
    sns.heatmap(role_means, annot=True, fmt=".2f",
                cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title("Mean Domain Scores by Participant Role")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "likert_role_heatmap.png", dpi=300)
    plt.close()
    print("   Saved: likert_role_heatmap.png")
    if role_means.max().max() - role_means.min().min() > 0.8:
        print("   THESIS FINDING: Role differences visually pronounced in domain scores")

# d) Correlation heatmap
fig, ax = plt.subplots(figsize=(max(10,len(likert_cols)*0.5),
                                max(8,len(likert_cols)*0.4)))
corr_matrix = quant_df[likert_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False,
            cmap="coolwarm", center=0, ax=ax,
            linewidths=0.3, vmin=-1, vmax=1)
ax.set_title("Item Correlation Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "likert_correlation_heatmap.png", dpi=300)
plt.close()
print("   Saved: likert_correlation_heatmap.png")


# ─────────────────────────────────────────────────────────────
# STEP 3 — CRONBACH'S ALPHA
# ─────────────────────────────────────────────────────────────
import pingouin as pg

print("\n── STEP 3: CRONBACH'S ALPHA ──")

def interpret_alpha(a):
    if a >= 0.90: return "Excellent"
    if a >= 0.80: return "Good"
    if a >= 0.70: return "Acceptable"
    if a >= 0.60: return "Questionable"
    return "Poor"

alpha_rows = []
all_acceptable = True
for d, items in domain_mapping.items():
    if len(items) < 2:
        continue
    try:
        result = pg.cronbach_alpha(quant_df[items])
        alpha  = result[0]
        ci     = result[1]
        interp = interpret_alpha(alpha)
        if alpha < 0.70:
            all_acceptable = False
        alpha_rows.append({
            "domain"        : d,
            "n_items"       : len(items),
            "alpha"         : round(alpha, 3),
            "ci_low"        : round(ci[0], 3),
            "ci_high"       : round(ci[1], 3),
            "interpretation": interp
        })
        flag = " ⚠️  REVIEW" if alpha < 0.60 else ""
        print(f"   {d:30s}: α={alpha:.3f} ({interp}){flag}")
    except Exception as e:
        print(f"   {d}: could not compute alpha — {e}")

alpha_df = pd.DataFrame(alpha_rows)
alpha_df.to_csv(OUTPUT_DIR + "cronbach_alpha_results.csv", index=False)
print("\n   Saved: cronbach_alpha_results.csv")

if all_acceptable and len(alpha_rows) > 0:
    print("   THESIS FINDING: All domains reached acceptable reliability (alpha >= 0.70)")


# ─────────────────────────────────────────────────────────────
# STEP 4 — EXPLORATORY FACTOR ANALYSIS
# ─────────────────────────────────────────────────────────────
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from factor_analyzer.factor_analyzer import calculate_kmo

print("\n── STEP 4: EXPLORATORY FACTOR ANALYSIS ──")

efa_data = quant_df[likert_cols].dropna()
min_n_for_efa = len(likert_cols) + 5

if len(efa_data) < min_n_for_efa:
    print(f"   ⚠️  Sample too small for full EFA "
          f"(n={len(efa_data)}, need ~{min_n_for_efa})")
    print("   Running with available data — interpret with caution")

# Step 4a — Factorability
try:
    chi2, p = calculate_bartlett_sphericity(efa_data)
    print(f"\n   Bartlett's Test: chi2={chi2:.2f}, p={p:.4f} "
          f"{'✅ PASS' if p < 0.05 else '❌ FAIL'}")
except Exception as e:
    print(f"   Bartlett's test skipped: {e}")

try:
    kmo_all, kmo_model = calculate_kmo(efa_data)
    kmo_interp = "Good" if kmo_model >= 0.80 else ("Acceptable" if kmo_model >= 0.60 else "Poor")
    print(f"   KMO Measure    : {kmo_model:.3f} ({kmo_interp}) "
          f"{'✅ PASS' if kmo_model >= 0.60 else '❌ FAIL'}")
except Exception as e:
    print(f"   KMO skipped: {e}")

# Step 4b — Scree plot & factor count
try:
    fa_eigen = FactorAnalyzer(n_factors=min(len(likert_cols), len(efa_data)-1),
                               rotation=None)
    fa_eigen.fit(efa_data)
    ev, v = fa_eigen.get_eigenvalues()
    n_factors = max(1, int(sum(ev > 1)))
    n_factors = min(n_factors, len(likert_cols)-1, len(efa_data)-2, 8)

    fig, ax = plt.subplots(figsize=(10,5))
    ax.plot(range(1, len(ev)+1), ev, "bo-", linewidth=2)
    ax.axhline(y=1, color="red", linestyle="--", label="Eigenvalue=1")
    ax.axvline(x=n_factors, color="green", linestyle="--",
               label=f"Suggested factors={n_factors}")
    ax.set_xlabel("Factor Number")
    ax.set_ylabel("Eigenvalue")
    ax.set_title("Scree Plot — Factor Analysis")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "efa_scree_plot.png", dpi=300)
    plt.close()
    print(f"\n   Suggested factors (eigenvalue>1): {n_factors}")
    print("   Saved: efa_scree_plot.png")
except Exception as e:
    n_factors = min(len(domain_names), 5)
    print(f"   Scree plot skipped: {e} — using {n_factors} factors")

# Step 4c — Run EFA
try:
    n_factors_run = min(n_factors, len(efa_data)-1, len(likert_cols)-1)
    fa = FactorAnalyzer(n_factors=n_factors_run, rotation="oblimin")
    fa.fit(efa_data)
    loadings_df = pd.DataFrame(
        fa.loadings_,
        index=likert_cols,
        columns=["Factor_"+str(i+1) for i in range(n_factors_run)]
    )
    communalities = pd.DataFrame({
        "item"         : likert_cols,
        "communality"  : fa.get_communalities()
    })

    print("\n   Factor Loadings (|loading| >= 0.40 marked):")
    print(f"   {'Item':8s}", end="")
    for col in loadings_df.columns:
        print(f"  {col:10s}", end="")
    print()
    for item in loadings_df.index:
        print(f"   {item:8s}", end="")
        for col in loadings_df.columns:
            val = loadings_df.loc[item, col]
            marker = " *" if abs(val) >= 0.40 else "  "
            print(f"  {val:+.3f}{marker}", end="")
        print()

    cross_loaders = []
    for item in loadings_df.index:
        high = (loadings_df.loc[item].abs() >= 0.40).sum()
        if high >= 2:
            cross_loaders.append(item)
    if cross_loaders:
        print(f"\n   ⚠️  Cross-loading items: {cross_loaders}")

    loadings_df.to_csv(OUTPUT_DIR + "efa_loadings.csv")
    communalities.to_csv(OUTPUT_DIR + "efa_communalities.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, max(6, len(likert_cols)*0.35)))
    sns.heatmap(loadings_df, annot=True, fmt=".2f",
                cmap="RdBu_r", center=0, ax=ax,
                linewidths=0.3, vmin=-1, vmax=1)
    ax.set_title("EFA Factor Loading Heatmap")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "efa_loading_heatmap.png", dpi=300)
    plt.close()
    print("\n   Saved: efa_loadings.csv, efa_communalities.csv")
    print("   Saved: efa_loading_heatmap.png")

except Exception as e:
    print(f"   EFA failed: {e}")
    loadings_df = pd.DataFrame()


# ─────────────────────────────────────────────────────────────
# STEP 5 — COMPOSITE SCORES
# ─────────────────────────────────────────────────────────────
print("\n── STEP 5: COMPOSITE SCORES ──")

for d, items in domain_mapping.items():
    valid_items = items[:]
    if not loadings_df.empty:
        high_loading = []
        for item in items:
            if item in loadings_df.index:
                if loadings_df.loc[item].abs().max() >= 0.40:
                    high_loading.append(item)
        if high_loading:
            valid_items = high_loading
    quant_df[d+"_score"] = quant_df[valid_items].mean(axis=1)
    print(f"   {d}_score: mean={quant_df[d+'_score'].mean():.2f}, "
          f"sd={quant_df[d+'_score'].std():.2f}, "
          f"items used={len(valid_items)}")

score_cols = [d+"_score" for d in domain_names]
composite_df = quant_df[["participant_id","role","institution"] + score_cols].copy() \
    if all(c in quant_df.columns for c in ["participant_id","role","institution"]) \
    else quant_df[score_cols].copy()
composite_df.to_csv(OUTPUT_DIR + "composite_scores.csv", index=False)
print("\n   Saved: composite_scores.csv")


# ─────────────────────────────────────────────────────────────
# STEP 6 — INFERENTIAL STATISTICS
# ─────────────────────────────────────────────────────────────
print("\n── STEP 6: INFERENTIAL STATISTICS ──")

inf_rows = []
if "role" in quant_df.columns:
    roles  = quant_df["role"].unique()
    groups = [quant_df[quant_df["role"]==r] for r in roles]

    for score in score_cols:
        grp_vals = [g[score].dropna().values for g in groups]
        grp_vals = [g for g in grp_vals if len(g) > 0]
        if len(grp_vals) < 2:
            continue
        try:
            h_stat, p_val = stats.kruskal(*grp_vals)
            sig = p_val < 0.05
            row = {
                "domain"     : score,
                "H_statistic": round(h_stat, 3),
                "p_value"    : round(p_val, 4),
                "significant": sig,
                "post_hoc"   : ""
            }
            if sig:
                try:
                    import scikit_posthocs as sp
                    dunn = sp.posthoc_dunn(
                        quant_df, val_col=score,
                        group_col="role", p_adjust="bonferroni"
                    )
                    sig_pairs = []
                    for i in range(len(roles)):
                        for j in range(i+1, len(roles)):
                            r1, r2 = roles[i], roles[j]
                            if r1 in dunn.index and r2 in dunn.columns:
                                if dunn.loc[r1,r2] < 0.05:
                                    sig_pairs.append(f"{r1} vs {r2}")
                    row["post_hoc"] = "; ".join(sig_pairs)
                except Exception:
                    row["post_hoc"] = "significant (install scikit-posthocs for pairs)"
                print(f"   THESIS FINDING: Significant role difference "
                      f"on {score} (H={h_stat:.2f}, p={p_val:.4f})")
            inf_rows.append(row)
            flag = " ✅" if sig else ""
            print(f"   {score:35s}: H={h_stat:.2f}, p={p_val:.4f}{flag}")
        except Exception as e:
            print(f"   {score}: test failed — {e}")

    inf_df = pd.DataFrame(inf_rows)
    inf_df.to_csv(OUTPUT_DIR + "inferential_results.csv", index=False)

    # Violin plots
    fig, axes = plt.subplots(
        1, len(score_cols),
        figsize=(max(16, len(score_cols)*3), 5)
    )
    if len(score_cols) == 1:
        axes = [axes]
    for ax, score in zip(axes, score_cols):
        if "role" in quant_df.columns:
            plot_df = quant_df[["role",score]].dropna()
            if not plot_df.empty:
                sns.violinplot(data=plot_df, x="role", y=score,
                               palette="Set2", ax=ax, inner="box")
                ax.set_title(score.replace("_score",""), fontsize=8)
                ax.set_xticklabels(ax.get_xticklabels(),
                                   rotation=30, ha="right", fontsize=7)
                ax.set_xlabel("")
    plt.suptitle("Domain Score Distributions by Role", y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "group_comparison_plots.png",
                dpi=300, bbox_inches="tight")
    plt.close()
    print("\n   Saved: inferential_results.csv")
    print("   Saved: group_comparison_plots.png")


# ─────────────────────────────────────────────────────────────
# STEP 7 — RANDOM FOREST
# ─────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

print("\n── STEP 7: RANDOM FOREST ──")

target_col = score_cols[0]
feature_cols = [c for c in score_cols if c != target_col]
print(f"   Target   : {target_col}")
print(f"   Features : {feature_cols}")

model_df = quant_df[feature_cols + [target_col]].copy()

if "role" in quant_df.columns:
    role_dummies = pd.get_dummies(quant_df["role"], prefix="role")
    model_df = pd.concat([model_df, role_dummies], axis=1)

if "institution" in quant_df.columns:
    inst_dummies = pd.get_dummies(quant_df["institution"], prefix="inst")
    model_df = pd.concat([model_df, inst_dummies], axis=1)

model_df = model_df.dropna()
X = model_df.drop(columns=[target_col])
y = model_df[target_col]
feature_names = list(X.columns)

if len(X) >= 10:
    stratify_col = None
    if "role" in quant_df.columns:
        role_aligned = quant_df.loc[model_df.index, "role"]
        if role_aligned.value_counts().min() >= 2:
            stratify_col = role_aligned

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42,
        stratify=stratify_col
    )

    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    y_pred_test  = rf.predict(X_test)
    y_pred_train = rf.predict(X_train)

    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_test  = mean_absolute_error(y_test, y_pred_test)
    r2_test   = r2_score(y_test, y_pred_test)
    r2_train  = r2_score(y_train, y_pred_train)

    cv_scores = cross_val_score(rf, X, y, cv=min(5, len(X)//2),
                                 scoring="r2")

    print(f"\n   Train R²     : {r2_train:.3f}")
    print(f"   Test  R²     : {r2_test:.3f}")
    print(f"   Test  RMSE   : {rmse_test:.3f}")
    print(f"   Test  MAE    : {mae_test:.3f}")
    print(f"   CV R² (mean) : {cv_scores.mean():.3f} "
          f"± {cv_scores.std():.3f}")

    if r2_test > 0.60:
        print(f"   THESIS FINDING: RF model R²={r2_test:.3f} > 0.60 "
              f"— strong predictive validity")

    fig, ax = plt.subplots(figsize=(7,6))
    ax.scatter(y_test, y_pred_test, alpha=0.7,
               color="#1A6B72", edgecolors="white", s=60)
    lims = [min(y_test.min(), y_pred_test.min())-0.1,
            max(y_test.max(), y_pred_test.max())+0.1]
    ax.plot(lims, lims, "r--", alpha=0.7, label="Perfect fit")
    ax.set_xlabel("Actual Score")
    ax.set_ylabel("Predicted Score")
    ax.set_title(f"RF: Actual vs Predicted\nR²={r2_test:.3f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "rf_actual_vs_predicted.png", dpi=300)
    plt.close()
    print("   Saved: rf_actual_vs_predicted.png")

    with open(OUTPUT_DIR + "rf_genai_readiness_model.pkl", "wb") as f:
        pickle.dump(rf, f)
    print("   Saved: rf_genai_readiness_model.pkl")

else:
    print("   ⚠️  Sample too small for train/test split — "
          "running full-data RF for SHAP only")
    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X, y)
    X_test = X.copy()
    feature_names = list(X.columns)
    r2_test = 0.0


# ─────────────────────────────────────────────────────────────
# STEP 8 — SHAP
# ─────────────────────────────────────────────────────────────
import shap

print("\n── STEP 8: SHAP EXPLAINABILITY ──")

try:
    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)

    shap_df = pd.DataFrame(shap_values, columns=feature_names)
    shap_df.to_csv(OUTPUT_DIR + "shap_values.csv", index=False)
    print("   Saved: shap_values.csv")

    # a) Beeswarm
    fig = plt.figure(figsize=(10,6))
    shap.summary_plot(shap_values, X_test,
                      feature_names=feature_names,
                      show=False, plot_type="dot")
    plt.title("SHAP Summary — Feature Impact on Target")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "shap_summary_beeswarm.png",
                dpi=300, bbox_inches="tight")
    plt.close()
    print("   Saved: shap_summary_beeswarm.png")

    # b) Bar importance
    fig = plt.figure(figsize=(10,6))
    shap.summary_plot(shap_values, X_test,
                      feature_names=feature_names,
                      show=False, plot_type="bar")
    plt.title("SHAP Feature Importance")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "shap_bar_importance.png",
                dpi=300, bbox_inches="tight")
    plt.close()
    print("   Saved: shap_bar_importance.png")

    # c) Dependence plots top 3
    mean_shap = np.abs(shap_values).mean(axis=0)
    top3_idx  = np.argsort(mean_shap)[::-1][:3]
    top3_feat = [feature_names[i] for i in top3_idx]
    top_feature = top3_feat[0]
    top_shap_val = mean_shap[top3_idx[0]]

    fig, axes = plt.subplots(1, 3, figsize=(15,5))
    for ax, feat in zip(axes, top3_feat):
        feat_idx = feature_names.index(feat)
        ax.scatter(X_test[feat].values,
                   shap_values[:, feat_idx],
                   alpha=0.6, color="#1A6B72", s=40)
        ax.axhline(0, color="red", linestyle="--", alpha=0.5)
        ax.set_xlabel(feat)
        ax.set_ylabel("SHAP value")
        ax.set_title(f"Dependence: {feat}")
    plt.suptitle("SHAP Dependence Plots — Top 3 Features")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "shap_dependence_top3.png", dpi=300)
    plt.close()
    print("   Saved: shap_dependence_top3.png")

    # d) Waterfall per role
    if "role" in quant_df.columns:
        roles_present = quant_df.loc[X_test.index, "role"].unique() \
            if X_test.index.isin(quant_df.index).all() \
            else quant_df["role"].unique()
        fig, axes = plt.subplots(
            1, min(3, len(roles_present)),
            figsize=(15, 5)
        )
        if len(roles_present) == 1:
            axes = [axes]
        for ax, role in zip(axes, roles_present[:3]):
            role_mask = quant_df["role"] == role
            role_idxs = quant_df.index[role_mask]
            test_mask = [i for i,idx in enumerate(X_test.index)
                         if idx in role_idxs]
            if test_mask:
                sample_i = test_mask[0]
                sv = shap_values[sample_i]
                sorted_idx = np.argsort(np.abs(sv))[::-1][:8]
                feat_subset = [feature_names[i] for i in sorted_idx]
                shap_subset = [sv[i] for i in sorted_idx]
                colors_wf   = ["#1A6B72" if s > 0 else "#D4820A"
                               for s in shap_subset]
                ax.barh(feat_subset, shap_subset,
                        color=colors_wf, alpha=0.8)
                ax.axvline(0, color="black", linewidth=0.8)
                ax.set_title(f"SHAP Waterfall\nRole: {role}", fontsize=9)
                ax.set_xlabel("SHAP value")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "shap_waterfall_sample.png", dpi=300)
        plt.close()
        print("   Saved: shap_waterfall_sample.png")

    print(f"\n   Top predictor: {top_feature} "
          f"(mean |SHAP| = {top_shap_val:.3f})")
    print(f"   Interpretation: '{top_feature}' is the strongest "
          f"driver of {target_col} in this sample")

except Exception as e:
    print(f"   ⚠️  SHAP failed: {e}")
    top_feature  = "N/A"
    top_shap_val = 0.0


# ─────────────────────────────────────────────────────────────
# STEP 9 — K-MEANS CLUSTERING
# ─────────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import umap

print("\n── STEP 9: K-MEANS CLUSTERING ──")

cluster_df = quant_df[score_cols].dropna().copy()
scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(cluster_df)

inertias    = []
sil_scores  = []
k_range     = range(2, min(7, len(cluster_df)))

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

# Elbow plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(k_range), inertias, "bo-", linewidth=2)
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Method")
ax2.plot(list(k_range), sil_scores, "ro-", linewidth=2)
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Scores")
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "kmeans_elbow.png", dpi=300)
plt.close()
print("   Saved: kmeans_elbow.png")

optimal_k = list(k_range)[np.argmax(sil_scores)]
print(f"   Optimal k (max silhouette): {optimal_k}")
print(f"   Silhouette scores: "
      f"{dict(zip(k_range, [round(s,3) for s in sil_scores]))}")

km_final = KMeans(n_clusters=optimal_k,
                  random_state=42, n_init=10)
km_final.fit(X_scaled)
cluster_df["cluster_id"] = km_final.labels_

# Cluster profiles
cluster_profiles = cluster_df.groupby("cluster_id")[score_cols].mean()
print("\n   Cluster profiles:")
print(cluster_profiles.round(2).to_string())

# ⚠️ ADJUST cluster names after reviewing profiles
cluster_names = {
    i: f"Cluster_{i+1}" for i in range(optimal_k)
}
if optimal_k >= 3:
    top_scores = cluster_profiles.mean(axis=1)
    sorted_cl  = top_scores.sort_values(ascending=False).index.tolist()
    preset_names = [
        "Strategic Adopters",
        "Cautious Economists",
        "Mission-Driven Resistors"
    ]
    for i, cl in enumerate(sorted_cl[:3]):
        cluster_names[cl] = preset_names[i]
elif optimal_k == 2:
    top_scores = cluster_profiles.mean(axis=1)
    high_cl = top_scores.idxmax()
    low_cl  = top_scores.idxmin()
    cluster_names[high_cl] = "GenAI Adopters"
    cluster_names[low_cl]  = "GenAI Hesitants"

cluster_df["cluster_name"] = cluster_df["cluster_id"].map(cluster_names)

# Merge back
quant_df = quant_df.merge(
    cluster_df[["cluster_id","cluster_name"]],
    left_index=True, right_index=True, how="left"
)

# a) Radar chart
fig, ax = plt.subplots(figsize=(8,8),
                        subplot_kw=dict(polar=True))
n_vars  = len(score_cols)
angles  = [i / float(n_vars) * 2 * np.pi
           for i in range(n_vars)]
angles += angles[:1]
ax.set_xticks(angles[:-1])
ax.set_xticklabels([s.replace("_score","") for s in score_cols],
                   size=8)
colors_radar = sns.color_palette("Set2", optimal_k)
for i, (cl_id, row) in enumerate(cluster_profiles.iterrows()):
    vals = row.values.tolist()
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2,
            color=colors_radar[i],
            label=cluster_names.get(cl_id, str(cl_id)))
    ax.fill(angles, vals, alpha=0.15, color=colors_radar[i])
ax.set_title("Cluster Profiles — Radar Chart", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3,1.1), fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "kmeans_radar_chart.png",
            dpi=300, bbox_inches="tight")
plt.close()
print("   Saved: kmeans_radar_chart.png")

# b) UMAP projection
try:
    reducer   = umap.UMAP(n_components=2, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    fig, ax   = plt.subplots(figsize=(9,7))
    scatter   = ax.scatter(
        embedding[:,0], embedding[:,1],
        c=cluster_df["cluster_id"].values,
        cmap="Set2", alpha=0.8, s=60, edgecolors="white"
    )
    for cl_id, cl_name in cluster_names.items():
        mask = cluster_df["cluster_id"] == cl_id
        cx   = embedding[mask, 0].mean()
        cy   = embedding[mask, 1].mean()
        ax.annotate(cl_name, (cx, cy),
                    fontsize=9, ha="center",
                    fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2",
                              fc="white", alpha=0.7))
    ax.set_title("UMAP Projection — Participant Clusters")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "kmeans_umap_projection.png", dpi=300)
    plt.close()
    print("   Saved: kmeans_umap_projection.png")
except Exception as e:
    print(f"   ⚠️  UMAP skipped: {e}")

# c) Role-cluster heatmap
if "role" in quant_df.columns and "cluster_id" in quant_df.columns:
    role_cluster = pd.crosstab(
        quant_df["cluster_name"].fillna("Unknown"),
        quant_df["role"]
    )
    fig, ax = plt.subplots(figsize=(8,4))
    sns.heatmap(role_cluster, annot=True, fmt="d",
                cmap="Blues", ax=ax, linewidths=0.5)
    ax.set_title("Cluster x Role Distribution")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "kmeans_role_cluster_heatmap.png", dpi=300)
    plt.close()
    print("   Saved: kmeans_role_cluster_heatmap.png")

cluster_profiles_out = cluster_profiles.copy()
cluster_profiles_out["cluster_name"] = [
    cluster_names.get(i,"") for i in cluster_profiles_out.index
]
cluster_profiles_out.to_csv(OUTPUT_DIR + "cluster_profiles.csv")

clustered_out = quant_df.copy()
if "participant_id" not in clustered_out.columns:
    clustered_out["participant_id"] = ["P"+str(i+1).zfill(3)
                                        for i in range(len(clustered_out))]
clustered_out.to_csv(OUTPUT_DIR + "clustered_participants.csv", index=False)
print("   Saved: cluster_profiles.csv")
print("   Saved: clustered_participants.csv")

print(f"\n   THESIS FINDING: {optimal_k} participant typologies identified")
for cl_id, cl_name in cluster_names.items():
    n = (cluster_df["cluster_id"] == cl_id).sum()
    print(f"     {cl_name}: n={n}")


# ─────────────────────────────────────────────────────────────
# STEP 10 — PREDICTIVE VALIDITY
# ─────────────────────────────────────────────────────────────
print("\n── STEP 10: PREDICTIVE VALIDITY ──")

validity_rows = []
if "cluster_id" in quant_df.columns:
    cluster_groups = [
        quant_df[quant_df["cluster_id"]==c]
        for c in quant_df["cluster_id"].unique()
    ]
    sig_count = 0
    for score in score_cols:
        grp_vals = [g[score].dropna().values for g in cluster_groups]
        grp_vals = [g for g in grp_vals if len(g) > 0]
        if len(grp_vals) < 2:
            continue
        try:
            h, p = stats.kruskal(*grp_vals)
            sig  = p < 0.05
            if sig:
                sig_count += 1
            validity_rows.append({
                "score"      : score,
                "H_statistic": round(h,3),
                "p_value"    : round(p,4),
                "significant": sig
            })
            flag = " ✅ significant" if sig else ""
            print(f"   {score:35s}: H={h:.2f}, p={p:.4f}{flag}")
        except Exception as e:
            print(f"   {score}: {e}")

    validity_df = pd.DataFrame(validity_rows)
    validity_df.to_csv(OUTPUT_DIR + "validity_check_results.csv", index=False)
    print("\n   Saved: validity_check_results.csv")

    if sig_count >= 3:
        print(f"   THESIS FINDING: Clusters validated — "
              f"significantly different on {sig_count} of "
              f"{len(score_cols)} domains")


# ─────────────────────────────────────────────────────────────
# STEP 11 — BLOCK 2 SUMMARY
# ─────────────────────────────────────────────────────────────
files_b2 = [
    "likert_item_means.png",
    "likert_domain_boxplots.png",
    "likert_role_heatmap.png",
    "likert_correlation_heatmap.png",
    "cronbach_alpha_results.csv",
    "efa_scree_plot.png",
    "efa_loadings.csv",
    "efa_communalities.csv",
    "efa_loading_heatmap.png",
    "composite_scores.csv",
    "inferential_results.csv",
    "group_comparison_plots.png",
    "rf_actual_vs_predicted.png",
    "rf_genai_readiness_model.pkl",
    "shap_values.csv",
    "shap_summary_beeswarm.png",
    "shap_bar_importance.png",
    "shap_dependence_top3.png",
    "shap_waterfall_sample.png",
    "kmeans_elbow.png",
    "kmeans_radar_chart.png",
    "kmeans_umap_projection.png",
    "kmeans_role_cluster_heatmap.png",
    "cluster_profiles.csv",
    "clustered_participants.csv",
    "validity_check_results.csv",
]
saved_b2 = [f for f in files_b2
            if os.path.exists(OUTPUT_DIR + f)]

validated_domains = [r["domain"] for r in alpha_rows
                     if r.get("alpha",0) >= 0.70] \
    if alpha_rows else []

print("\n" + "="*55)
print("  BLOCK 2 COMPLETE — QUANTITATIVE ANALYSIS")
print("="*55)
print(f"  Participants analyzed     : {len(quant_df)}")
print(f"  Items analyzed            : {len(likert_cols)}")
print(f"  Domains validated (≥0.70) : {len(validated_domains)}")
print(f"  EFA factors extracted     : {n_factors_run if 'n_factors_run' in dir() else 'N/A'}")
print(f"  RF Model R²               : {r2_test:.3f}")
print(f"  Top SHAP predictor        : {top_feature}")
print(f"  K-Means clusters (k)      : {optimal_k}")
print(f"  Cluster names             : {list(cluster_names.values())}")
print(f"  Files saved               : {len(saved_b2)} of {len(files_b2)}")
print("-"*55)
print("  FILES SAVED:")
for f in saved_b2:
    print(f"    ✅ {f}")
missing_b2 = [f for f in files_b2 if f not in saved_b2]
for f in missing_b2:
    print(f"    ⚠️  {f} — not generated")
print("="*55)
print("  NEXT: Run Block 3 — Mixed Methods Integration")
print("  (Convergence Matrix → Joint Display → Meta-Inference)")
print("="*55)

✅ Block 2 dependencies ready

── STEP 1: LOAD & VALIDATE ──
   Shape  : (86, 44)
   Columns: ['Timestamp', "How important is GenAI for maintaining your medical center's competitive edge?Ê", 'Q1', 'How likely is it that GenAI will lead to significant innovations in patient care?Ê', 'What percentage of your strategic planning involves the use of GenAI?', "Rate the impact of GenAI on your medical center's market positioningÊ", 'How ready is your organization to adopt GenAI?', 'Rate the overall attitude of healthcare staff towards GenAI adoptionÊ', 'How often are GenAI topics included in staff meetings or training sessions?Ê', 'Q2'] ...

   Participants by role:
role
executive    86

   Likert items detected: 13
   Scale range detected  : 1 to 5

   Missing after imputation: 0

   Domains (5): ['DOMAIN_1', 'DOMAIN_2', 'DOMAIN_3', 'DOMAIN_4', 'DOMAIN_5']

── STEP 2: DESCRIPTIVE STATISTICS ──
     mean   std  min  max  skewness  kurtosis
Q1   3.41  1.10  1.0  5.0     -0.60     -0.51
Q2   4.0

In [8]:
# ═════════════════════════════════════════════════════════════
# BLOCK 3 — MIXED METHODS INTEGRATION
# ═════════════════════════════════════════════════════════════

import os, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
import seaborn as sns
import networkx as nx
warnings.filterwarnings("ignore")

OUTPUT_DIR = "/content/thesis_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Block 3 starting — Mixed Methods Integration")


# ─────────────────────────────────────────────────────────────
# PREREQUISITE CHECK
# ─────────────────────────────────────────────────────────────
required_files = [
    "bertopic_document_topics.csv",
    "bertopic_construct_mapping.csv",
    "sentiment_by_role_topic.csv",
    "cooccurrence_edges.csv",
    "manual_vs_ml_comparison_template.csv",
    "composite_scores.csv",
    "clustered_participants.csv",
    "cluster_profiles.csv",
    "shap_values.csv",
    "cronbach_alpha_results.csv",
]
print("\n── PREREQUISITE CHECK ──")
all_present = True
for f in required_files:
    path   = OUTPUT_DIR + f
    exists = os.path.exists(path)
    flag   = "✅" if exists else "❌ MISSING"
    print(f"   {flag}  {f}")
    if not exists:
        all_present = False

if not all_present:
    print("\n⚠️  Some files missing — run Blocks 1 and 2 first.")
    print("   Continuing with available files...")


# ─────────────────────────────────────────────────────────────
# LOAD ALL BLOCK 1 + 2 OUTPUTS
# ─────────────────────────────────────────────────────────────
def safe_load(fname, fallback_cols=None):
    path = OUTPUT_DIR + fname
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            return df if not df.empty else pd.DataFrame(columns=fallback_cols or [])
        except Exception:
            print(f"   ⚠️  {fname} empty or unreadable — using empty DataFrame")
            return pd.DataFrame(columns=fallback_cols or [])
    print(f"   ⚠️  {fname} not found — using empty DataFrame")
    return pd.DataFrame(columns=fallback_cols or [])

doc_topics_df    = safe_load("bertopic_document_topics.csv")
construct_map_df = safe_load("bertopic_construct_mapping.csv")
sentiment_df     = safe_load("sentiment_by_role_topic.csv")
edges_df         = safe_load("cooccurrence_edges.csv")
manual_df        = safe_load("manual_vs_ml_comparison_template.csv")
composite_df     = safe_load("composite_scores.csv")
clustered_df     = safe_load("clustered_participants.csv")
cluster_prof_df  = safe_load("cluster_profiles.csv")
shap_df          = safe_load("shap_values.csv")
alpha_df         = safe_load("cronbach_alpha_results.csv")
inferential_df   = safe_load("inferential_results.csv")

# Detect score columns
score_cols = [c for c in clustered_df.columns if c.endswith("_score")]
domain_names = [c.replace("_score","") for c in score_cols]
print(f"\n   Score columns detected: {score_cols}")


# ─────────────────────────────────────────────────────────────
# STEP 1 — MASTER DATASET
# ─────────────────────────────────────────────────────────────
print("\n── STEP 1: MASTER DATASET ──")

master_df = clustered_df.copy()

# Ensure participant_id exists
if "participant_id" not in master_df.columns:
    master_df["participant_id"] = ["P"+str(i+1).zfill(3)
                                    for i in range(len(master_df))]

# Merge Block 1 document topics
if not doc_topics_df.empty and "participant_id" in doc_topics_df.columns:
    doc_top_agg = doc_topics_df.sort_values(
        "topic_probability", ascending=False
    ).groupby("participant_id").first().reset_index()
    doc_top_agg = doc_top_agg.rename(columns={
        "topic_id"         : "primary_topic_id",
        "topic_probability": "primary_topic_prob"
    })
    master_df = master_df.merge(
        doc_top_agg[["participant_id","primary_topic_id","primary_topic_prob"]],
        on="participant_id", how="left"
    )

# Add primary DBA construct from construct mapping
if not construct_map_df.empty and "topic_id" in construct_map_df.columns:
    construct_lookup = construct_map_df.set_index("topic_id")[
        "primary_construct"
    ].to_dict()
    if "primary_topic_id" in master_df.columns:
        master_df["primary_dba_construct"] = master_df[
            "primary_topic_id"
        ].map(construct_lookup)

# Merge sentiment
if not sentiment_df.empty:
    if "participant_id" in sentiment_df.columns:
        sent_agg = sentiment_df.groupby("participant_id")[
            "sentiment_compound"
        ].mean().reset_index()
        master_df = master_df.merge(sent_agg, on="participant_id", how="left")
    elif "role" in sentiment_df.columns:
        sent_role = sentiment_df.groupby("role")[
            "sentiment_compound"
        ].mean().to_dict()
        if "role" in master_df.columns:
            master_df["sentiment_compound"] = master_df["role"].map(sent_role)

if "sentiment_compound" not in master_df.columns:
    master_df["sentiment_compound"] = 0.0

master_df.to_csv(OUTPUT_DIR + "master_integrated_dataset.csv", index=False)
print(f"   Master dataset shape: {master_df.shape}")
print(f"   Columns: {list(master_df.columns)}")
print(f"   Saved: master_integrated_dataset.csv")


# ─────────────────────────────────────────────────────────────
# STEP 2 — CONVERGENCE MATRIX
# ─────────────────────────────────────────────────────────────
print("\n── STEP 2: CONVERGENCE MATRIX ──")

# Build construct list from available sources
all_constructs = set()
if not construct_map_df.empty and "primary_construct" in construct_map_df.columns:
    all_constructs.update(construct_map_df["primary_construct"].dropna().unique())
for d in domain_names:
    all_constructs.add(d)
all_constructs = sorted(all_constructs)

# Alpha lookup
alpha_lookup = {}
if not alpha_df.empty and "domain" in alpha_df.columns:
    for _, row in alpha_df.iterrows():
        alpha_lookup[row["domain"]] = row.get("alpha", np.nan)

# Composite score lookup
composite_means = {}
composite_sds   = {}
for sc in score_cols:
    if sc in master_df.columns:
        composite_means[sc.replace("_score","")] = master_df[sc].mean()
        composite_sds[sc.replace("_score","")]   = master_df[sc].std()

# SHAP rank lookup
shap_ranks = {}
if not shap_df.empty:
    mean_shap = shap_df.abs().mean()
    sorted_shap = mean_shap.sort_values(ascending=False)
    for rank, feat in enumerate(sorted_shap.index):
        clean = feat.replace("_score","")
        shap_ranks[clean] = rank + 1

# Sentiment per construct
sent_by_construct = {}
if "primary_dba_construct" in master_df.columns:
    sent_by_construct = master_df.groupby(
        "primary_dba_construct"
    )["sentiment_compound"].mean().to_dict()

# Convergence status from manual template
conv_status_lookup = {}
if not manual_df.empty and "convergence_status" in manual_df.columns:
    if "dba_construct" in manual_df.columns:
        for _, row in manual_df.iterrows():
            status = str(row.get("convergence_status","")).strip()
            if status and status.lower() not in ["nan","","none"]:
                conv_status_lookup[row["dba_construct"]] = status

conv_rows = []
for construct in all_constructs:
    alpha_val   = alpha_lookup.get(construct, np.nan)
    comp_mean   = composite_means.get(construct, np.nan)
    comp_sd     = composite_sds.get(construct, np.nan)
    shap_rank   = shap_ranks.get(construct, np.nan)
    sent_mean   = sent_by_construct.get(construct, np.nan)
    conv_status = conv_status_lookup.get(construct, "Partial")

    # Auto-compute convergence score
    score = 0
    if not np.isnan(alpha_val) and alpha_val >= 0.70:
        score += 1
    if not np.isnan(comp_mean) and comp_mean >= 3.0:
        score += 1
    if not np.isnan(sent_mean) and sent_mean >= 0:
        score += 1
    if conv_status == "Confirmed":
        score = 3
    elif conv_status == "Divergent":
        score = 1

    conv_rows.append({
        "dba_construct"      : construct,
        "cronbach_alpha"     : round(alpha_val,3) if not np.isnan(alpha_val) else "N/A",
        "mean_composite"     : round(comp_mean,2) if not np.isnan(comp_mean) else "N/A",
        "sd_composite"       : round(comp_sd,2)   if not np.isnan(comp_sd)   else "N/A",
        "shap_rank"          : int(shap_rank)      if not np.isnan(shap_rank) else "N/A",
        "sentiment_mean"     : round(sent_mean,3)  if not np.isnan(sent_mean) else "N/A",
        "convergence_status" : conv_status,
        "convergence_score"  : score,
    })

conv_df = pd.DataFrame(conv_rows)
conv_df = conv_df.sort_values("convergence_score", ascending=False)
conv_df.to_csv(OUTPUT_DIR + "convergence_matrix_full.csv", index=False)

print("\n   Convergence Matrix:")
print(conv_df[["dba_construct","convergence_score",
               "convergence_status","mean_composite",
               "sentiment_mean"]].to_string(index=False))

for _, row in conv_df.iterrows():
    if row["convergence_score"] == 3:
        print(f"   THESIS FINDING: '{row['dba_construct']}' CONFIRMED across all streams")
    if row["convergence_score"] == 1:
        print(f"   THESIS FINDING: '{row['dba_construct']}' DIVERGENT — analytically valuable")

# Convergence matrix visualization
status_colors = {"Confirmed":"#2E7D32","Partial":"#F57F17","Divergent":"#C62828"}
row_colors = [status_colors.get(r["convergence_status"],"#888888")
              for _, r in conv_df.iterrows()]

fig, ax = plt.subplots(figsize=(14, max(4, len(conv_df)*0.6)))
ax.axis("off")
display_cols = ["dba_construct","convergence_score","convergence_status",
                "mean_composite","cronbach_alpha","shap_rank","sentiment_mean"]
available_cols = [c for c in display_cols if c in conv_df.columns]
table_data  = [conv_df[available_cols].columns.tolist()]
table_data += conv_df[available_cols].values.tolist()

tbl = ax.table(
    cellText  = [row for row in table_data[1:]],
    colLabels = table_data[0],
    cellLoc   = "center",
    loc       = "center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.auto_set_column_width(col=list(range(len(available_cols))))

for i, (_, row) in enumerate(conv_df.iterrows()):
    bg = status_colors.get(row["convergence_status"],"#EEEEEE")
    for j in range(len(available_cols)):
        tbl[i+1, j].set_facecolor(bg + "55")

for j in range(len(available_cols)):
    tbl[0, j].set_facecolor("#1B2A4A")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

ax.set_title("Convergence Matrix — Three-Source Validation",
             fontsize=11, fontweight="bold", pad=10)
legend_patches = [mpatches.Patch(color=c, label=l)
                  for l,c in status_colors.items()]
ax.legend(handles=legend_patches, loc="lower center",
          bbox_to_anchor=(0.5,-0.05), ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "convergence_matrix.png",
            dpi=300, bbox_inches="tight")
plt.close()
print("\n   Saved: convergence_matrix_full.csv")
print("   Saved: convergence_matrix.png")


# ─────────────────────────────────────────────────────────────
# STEP 3 — JOINT DISPLAY TABLE
# ─────────────────────────────────────────────────────────────
print("\n── STEP 3: JOINT DISPLAY TABLE ──")

# Dominant cluster per construct
cluster_by_construct = {}
if "primary_dba_construct" in master_df.columns and "cluster_name" in master_df.columns:
    for construct in all_constructs:
        subset = master_df[master_df["primary_dba_construct"]==construct]
        if not subset.empty and "cluster_name" in subset.columns:
            dominant = subset["cluster_name"].value_counts().idxmax()
            cluster_by_construct[construct] = dominant

# BERTopic top words per construct
bertopic_words_by_construct = {}
if not construct_map_df.empty and "primary_construct" in construct_map_df.columns:
    for construct in all_constructs:
        subset = construct_map_df[
            construct_map_df["primary_construct"]==construct
        ]
        if not subset.empty and "top_words" in subset.columns:
            bertopic_words_by_construct[construct] = subset["top_words"].iloc[0]

jd_rows = []
for _, row in conv_df.iterrows():
    construct = row["dba_construct"]
    sc        = construct + "_score"
    mean_val  = composite_means.get(construct, np.nan)
    sd_val    = composite_sds.get(construct, np.nan)
    quant_str = f"{mean_val:.2f} ± {sd_val:.2f}" \
                if not np.isnan(mean_val) else "N/A"

    jd_rows.append({
        "dba_construct"       : construct,
        "qualitative_evidence": "[QUOTE_1] | [QUOTE_2]",
        "quantitative_score"  : quant_str,
        "bertopic_top_words"  : bertopic_words_by_construct.get(construct,"N/A"),
        "cluster_prevalence"  : cluster_by_construct.get(construct,"N/A"),
        "shap_rank"           : row["shap_rank"],
        "sentiment_mean"      : row["sentiment_mean"],
        "convergence_status"  : row["convergence_status"],
        "meta_inference"      : "[TO BE WRITTEN]",
    })

jd_df = pd.DataFrame(jd_rows)
jd_df.to_csv(OUTPUT_DIR + "joint_display_template.csv", index=False)

fig, ax = plt.subplots(figsize=(18, max(5, len(jd_df)*0.8)))
ax.axis("off")
disp_cols = ["dba_construct","quantitative_score","shap_rank",
             "sentiment_mean","cluster_prevalence","convergence_status"]
avail_jd  = [c for c in disp_cols if c in jd_df.columns]

tbl2 = ax.table(
    cellText  = jd_df[avail_jd].values.tolist(),
    colLabels = avail_jd,
    cellLoc   = "center",
    loc       = "center"
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(7.5)
tbl2.auto_set_column_width(col=list(range(len(avail_jd))))

for i, (_, row) in enumerate(jd_df.iterrows()):
    bg = status_colors.get(row["convergence_status"],"#EEEEEE")
    for j in range(len(avail_jd)):
        tbl2[i+1,j].set_facecolor(bg+"33")

for j in range(len(avail_jd)):
    tbl2[0,j].set_facecolor("#1A6B72")
    tbl2[0,j].set_text_props(color="white", fontweight="bold")

ax.set_title("Joint Display Table — Mixed Methods Integration\n"
             "(Replace [QUOTE] placeholders with verbatim participant quotes "
             "from NVivo/Atlas.ti coding)",
             fontsize=9, pad=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "joint_display_table.png",
            dpi=300, bbox_inches="tight")
plt.close()
print("   Saved: joint_display_template.csv")
print("   Saved: joint_display_table.png")
print("\n   NOTE: Fill [QUOTE_1] and [QUOTE_2] placeholders with")
print("   anonymized participant quotes after NVivo/Atlas.ti coding.")
print("   Fill [TO BE WRITTEN] with meta-inference sentence per construct.")


# ─────────────────────────────────────────────────────────────
# STEP 4 — CLUSTER x THEME CROSS-ANALYSIS
# ─────────────────────────────────────────────────────────────
print("\n── STEP 4: CLUSTER x THEME CROSS-ANALYSIS ──")

cluster_theme_rows = []
if "cluster_name" in master_df.columns:
    clusters = master_df["cluster_name"].dropna().unique()

    # a) Cluster x construct heatmap
    if score_cols and all(sc in master_df.columns for sc in score_cols):
        clust_scores = master_df.groupby("cluster_name")[score_cols].mean()
        clust_scores.columns = domain_names
        fig, ax = plt.subplots(figsize=(10, max(4,len(clusters)*0.8)))
        sns.heatmap(clust_scores, annot=True, fmt=".2f",
                    cmap="YlOrRd", ax=ax, linewidths=0.5)
        ax.set_title("Cluster x Domain Score Heatmap")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "cluster_theme_heatmap.png", dpi=300)
        plt.close()
        print("   Saved: cluster_theme_heatmap.png")

    # b) Sentiment per cluster
    if "sentiment_compound" in master_df.columns:
        sent_cluster = master_df.groupby("cluster_name")[
            "sentiment_compound"
        ].mean().reset_index()
        fig, ax = plt.subplots(figsize=(8,4))
        colors_cl = sns.color_palette("Set2", len(sent_cluster))
        ax.bar(sent_cluster["cluster_name"],
               sent_cluster["sentiment_compound"],
               color=colors_cl, alpha=0.85)
        ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_title("Mean Sentiment by Cluster")
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Compound Sentiment")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "cluster_sentiment_barplot.png", dpi=300)
        plt.close()
        print("   Saved: cluster_sentiment_barplot.png")

    # c) SHAP profile per cluster
    if not shap_df.empty and "cluster_name" in master_df.columns:
        try:
            shap_aligned = shap_df.copy()
            shap_aligned.index = master_df.index[:len(shap_aligned)]
            shap_with_cluster = shap_aligned.join(
                master_df[["cluster_name"]].iloc[:len(shap_aligned)]
            )
            shap_cluster_mean = shap_with_cluster.groupby(
                "cluster_name"
            ).mean()
            top3_feats = shap_df.abs().mean().sort_values(
                ascending=False
            ).head(3).index.tolist()
            available_top3 = [f for f in top3_feats
                               if f in shap_cluster_mean.columns]
            if available_top3:
                fig, ax = plt.subplots(figsize=(10,5))
                shap_cluster_mean[available_top3].plot(
                    kind="bar", ax=ax, colormap="Set2", alpha=0.85
                )
                ax.set_title("Top 3 SHAP Features by Cluster")
                ax.set_xlabel("Cluster")
                ax.set_ylabel("Mean SHAP Value")
                ax.set_xticklabels(ax.get_xticklabels(),
                                   rotation=20, ha="right")
                ax.legend(bbox_to_anchor=(1.05,1), loc="upper left",
                          fontsize=8)
                plt.tight_layout()
                plt.savefig(OUTPUT_DIR + "cluster_shap_profile.png", dpi=300)
                plt.close()
                print("   Saved: cluster_shap_profile.png")
        except Exception as e:
            print(f"   ⚠️  SHAP cluster plot skipped: {e}")

    # Cluster narrative
    for cl in clusters:
        subset = master_df[master_df["cluster_name"]==cl]
        dom_construct = "N/A"
        if "primary_dba_construct" in subset.columns:
            vc = subset["primary_dba_construct"].value_counts()
            if not vc.empty:
                dom_construct = vc.idxmax()
        sent_val = subset["sentiment_compound"].mean() \
            if "sentiment_compound" in subset.columns else 0
        sent_dir  = "positive" if sent_val > 0.05 else \
                    ("negative" if sent_val < -0.05 else "neutral")
        top_domain = "N/A"
        if score_cols and all(sc in subset.columns for sc in score_cols):
            top_domain = subset[score_cols].mean().idxmax()
        print(f"\n   Cluster '{cl}':")
        print(f"     Dominant construct : {dom_construct}")
        print(f"     Sentiment          : {sent_dir} ({sent_val:.3f})")
        print(f"     Highest domain     : {top_domain}")
        print(f"     n participants     : {len(subset)}")

        cluster_theme_rows.append({
            "cluster_name"        : cl,
            "n_participants"      : len(subset),
            "dominant_construct"  : dom_construct,
            "sentiment_direction" : sent_dir,
            "sentiment_mean"      : round(sent_val,3),
            "highest_domain"      : top_domain,
        })
        if abs(sent_val) > 0.1:
            print(f"   THESIS FINDING: Cluster '{cl}' shows {sent_dir} "
                  f"sentiment ({sent_val:.3f}) on GenAI discourse")

cluster_theme_df = pd.DataFrame(cluster_theme_rows)
cluster_theme_df.to_csv(OUTPUT_DIR + "cluster_theme_profiles.csv", index=False)
print("\n   Saved: cluster_theme_profiles.csv")


# ─────────────────────────────────────────────────────────────
# STEP 5 — ROLE-LEVEL INTEGRATION
# ─────────────────────────────────────────────────────────────
print("\n── STEP 5: ROLE-LEVEL INTEGRATION ──")

role_rows = []
if "role" in master_df.columns:
    roles = master_df["role"].dropna().unique()

    # a) Radar chart per role
    if score_cols and all(sc in master_df.columns for sc in score_cols):
        role_means = master_df.groupby("role")[score_cols].mean()
        n_vars = len(score_cols)
        angles = [n/float(n_vars)*2*np.pi*i for i in range(n_vars)]
        angles += angles[:1]
        fig, ax = plt.subplots(figsize=(8,8),
                                subplot_kw=dict(polar=True))
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([s.replace("_score","") for s in score_cols],
                           size=8)
        colors_r = sns.color_palette("Set1", len(roles))
        for i, role in enumerate(roles):
            if role not in role_means.index:
                continue
            vals = role_means.loc[role].values.tolist()
            vals += vals[:1]
            ax.plot(angles, vals, "o-", linewidth=2,
                    color=colors_r[i], label=str(role))
            ax.fill(angles, vals, alpha=0.1, color=colors_r[i])
        ax.set_title("Domain Score Profile by Role", pad=20)
        ax.legend(loc="upper right",
                  bbox_to_anchor=(1.3,1.1), fontsize=9)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "role_integration_radar.png",
                    dpi=300, bbox_inches="tight")
        plt.close()
        print("   Saved: role_integration_radar.png")

    # b) Role x construct sentiment matrix
    if "primary_dba_construct" in master_df.columns and \
       "sentiment_compound" in master_df.columns:
        try:
            role_sent = master_df.pivot_table(
                values="sentiment_compound",
                index="role",
                columns="primary_dba_construct",
                aggfunc="mean"
            )
            if not role_sent.empty:
                fig, ax = plt.subplots(figsize=(12,4))
                sns.heatmap(role_sent, annot=True, fmt=".2f",
                            cmap="RdYlGn", center=0, ax=ax,
                            linewidths=0.5)
                ax.set_title("Sentiment by Role x DBA Construct")
                plt.tight_layout()
                plt.savefig(OUTPUT_DIR + "role_sentiment_topic_matrix.png",
                            dpi=300)
                plt.close()
                print("   Saved: role_sentiment_topic_matrix.png")
        except Exception as e:
            print(f"   ⚠️  Role sentiment matrix skipped: {e}")

    # c) Sankey diagram
    try:
        import subprocess
        subprocess.check_call(
            ["pip","install","plotly","kaleido","-q"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        import plotly.graph_objects as go

        if "cluster_name" in master_df.columns and \
           "primary_dba_construct" in master_df.columns:
            roles_list     = master_df["role"].dropna().unique().tolist()
            clusters_list  = master_df["cluster_name"].dropna().unique().tolist()
            constructs_list= master_df["primary_dba_construct"].dropna().unique().tolist()
            all_nodes = roles_list + clusters_list + constructs_list
            node_idx  = {n:i for i,n in enumerate(all_nodes)}

            sources, targets, values = [], [], []
            for role in roles_list:
                for cl in clusters_list:
                    n = len(master_df[
                        (master_df["role"]==role) &
                        (master_df["cluster_name"]==cl)
                    ])
                    if n > 0:
                        sources.append(node_idx[role])
                        targets.append(node_idx[cl])
                        values.append(n)

            for cl in clusters_list:
                for con in constructs_list:
                    n = len(master_df[
                        (master_df["cluster_name"]==cl) &
                        (master_df["primary_dba_construct"]==con)
                    ])
                    if n > 0:
                        sources.append(node_idx[cl])
                        targets.append(node_idx[con])
                        values.append(n)

            if sources:
                fig_s = go.Figure(go.Sankey(
                    node=dict(
                        pad=15, thickness=20,
                        label=all_nodes,
                        color=["#1B2A4A"]*len(roles_list) +
                              ["#1A6B72"]*len(clusters_list) +
                              ["#D4820A"]*len(constructs_list)
                    ),
                    link=dict(source=sources, target=targets,
                              value=values)
                ))
                fig_s.update_layout(
                    title_text="Role → Cluster → DBA Construct Flow",
                    font_size=10, height=500
                )
                fig_s.write_image(OUTPUT_DIR + "role_cluster_sankey.png",
                                  scale=2)
                print("   Saved: role_cluster_sankey.png")
    except Exception as e:
        print(f"   ⚠️  Sankey skipped: {e}")
        fig, ax = plt.subplots(figsize=(10,6))
        if "cluster_name" in master_df.columns:
            role_cluster = pd.crosstab(
                master_df["role"].fillna("Unknown"),
                master_df["cluster_name"].fillna("Unknown")
            )
            sns.heatmap(role_cluster, annot=True, fmt="d",
                        cmap="Blues", ax=ax)
            ax.set_title("Role x Cluster Distribution\n(Sankey fallback)")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR + "role_cluster_sankey.png", dpi=300)
        plt.close()
        print("   Saved: role_cluster_sankey.png (heatmap fallback)")

    # Role profiles table
    for role in roles:
        subset    = master_df[master_df["role"]==role]
        dom_c     = "N/A"
        if "primary_dba_construct" in subset.columns:
            vc = subset["primary_dba_construct"].value_counts()
            if not vc.empty:
                dom_c = vc.idxmax()
        sent_val  = subset["sentiment_compound"].mean() \
            if "sentiment_compound" in subset.columns else 0
        dom_cl    = "N/A"
        if "cluster_name" in subset.columns:
            vc2 = subset["cluster_name"].value_counts()
            if not vc2.empty:
                dom_cl = vc2.idxmax()
        top_dom   = "N/A"
        if score_cols and all(sc in subset.columns for sc in score_cols):
            top_dom = subset[score_cols].mean().idxmax()
        role_rows.append({
            "role"               : role,
            "n_participants"     : len(subset),
            "dominant_construct" : dom_c,
            "mean_sentiment"     : round(sent_val,3),
            "dominant_cluster"   : dom_cl,
            "highest_domain"     : top_dom,
        })

    role_int_df = pd.DataFrame(role_rows)
    role_int_df.to_csv(OUTPUT_DIR + "role_integration_profiles.csv",
                       index=False)
    print("\n   Role Integration Profiles:")
    print(role_int_df.to_string(index=False))
    print("\n   Saved: role_integration_profiles.csv")

    if len(role_int_df) >= 2:
        constructs_per_role = role_int_df["dominant_construct"].nunique()
        if constructs_per_role >= 2:
            print("   THESIS FINDING: Roles show distinct DBA construct "
                  "pathways toward GenAI adoption")


# ─────────────────────────────────────────────────────────────
# STEP 6 — LEBANON CONTEXTUAL LAYER
# ─────────────────────────────────────────────────────────────
print("\n── STEP 6: LEBANON CONTEXTUAL LAYER ──")

lb_terms = [
    "moph","syndicate","nssf","crisis","dollar","lira",
    "brain","drain","accreditation","jci","private","public",
    "nonprofit","university","aubmc","lau","makassed","reform",
    "inflation","shortage","ministry","lebanese"
]
# ⚠️ ADJUST these terms to match your actual transcript vocabulary

lb_rows = []
G_lb = nx.Graph()

if not edges_df.empty and "word1" in edges_df.columns:
    for _, row in edges_df.iterrows():
        w1 = str(row["word1"]).lower()
        w2 = str(row["word2"]).lower()
        wt = row.get("weight", 1)
        is_lb = any(t in w1 or t in w2 for t in lb_terms)
        if is_lb:
            lb_rows.append({"word1":w1,"word2":w2,"weight":wt})
            if G_lb.has_edge(w1,w2):
                G_lb[w1][w2]["weight"] += wt
            else:
                G_lb.add_edge(w1, w2, weight=wt)

lb_context_df = pd.DataFrame(lb_rows)
lb_context_df.to_csv(OUTPUT_DIR + "lebanon_context_analysis.csv", index=False)

if len(G_lb.nodes) > 1:
    lb_node_colors = []
    for node in G_lb.nodes():
        is_lb_node = any(t in node for t in lb_terms)
        lb_node_colors.append("#C62828" if is_lb_node else "#1A6B72")
    degree_dict = dict(G_lb.degree())
    node_sizes  = [300 + degree_dict.get(n,1)*200 for n in G_lb.nodes()]

    fig, ax = plt.subplots(figsize=(12,10))
    pos = nx.spring_layout(G_lb, seed=42, k=2)
    weights = [G_lb[u][v].get("weight",1) for u,v in G_lb.edges()]
    nx.draw_networkx_nodes(G_lb, pos, node_color=lb_node_colors,
                           node_size=node_sizes, alpha=0.9, ax=ax)
    nx.draw_networkx_edges(G_lb, pos,
                           width=[w*0.3 for w in weights],
                           alpha=0.5, edge_color="#AAAAAA", ax=ax)
    nx.draw_networkx_labels(G_lb, pos, font_size=8, ax=ax)
    ax.set_title("GenAI Discourse in Lebanese AMC Context\n"
                 "(Red nodes = Lebanon-specific terms)",
                 fontsize=11)
    ax.axis("off")
    lb_legend = [
        mpatches.Patch(color="#C62828", label="Lebanon-specific"),
        mpatches.Patch(color="#1A6B72", label="GenAI/Healthcare")
    ]
    ax.legend(handles=lb_legend, loc="lower left", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + "lebanon_context_network.png", dpi=300)
    plt.close()
    print("   Saved: lebanon_context_network.png")

    if not lb_context_df.empty:
        lb_word_freq = pd.concat(
            [lb_context_df["word1"], lb_context_df["word2"]]
        ).value_counts()
        top_lb = lb_word_freq.index[0] if not lb_word_freq.empty else "N/A"
        print(f"\n   THESIS FINDING: In Lebanese AMC context, GenAI "
              f"discourse co-occurs most with '{top_lb}'")
        print(f"   Lebanon-specific edges found: {len(lb_context_df)}")
else:
    print("   ⚠️  No Lebanon-specific co-occurrences found in edge data")
    print("   This may indicate transcripts lack explicit Lebanon-specific")
    print("   terminology — review lb_terms list and transcript content")
    lb_context_df = pd.DataFrame(columns=["word1","word2","weight"])
    lb_context_df.to_csv(OUTPUT_DIR + "lebanon_context_analysis.csv",
                         index=False)

print("   Saved: lebanon_context_analysis.csv")


# ─────────────────────────────────────────────────────────────
# STEP 7 — META-INFERENCE ENGINE
# ─────────────────────────────────────────────────────────────
print("\n── STEP 7: META-INFERENCE ENGINE ──")

# Weights — justify in Chapter 3
W_CONV  = 0.35
W_COMP  = 0.25
W_SENT  = 0.20
W_SHAP  = 0.20
# ⚠️ ADJUST weights based on your methodological priority

meta_rows = []
comp_vals = [v for v in composite_means.values()
             if not np.isnan(v)]
comp_min  = min(comp_vals) if comp_vals else 1
comp_max  = max(comp_vals) if comp_vals else 5
shap_vals = [v for v in shap_ranks.values()
             if not np.isnan(v)]
shap_max  = max(shap_vals) if shap_vals else 1

for _, row in conv_df.iterrows():
    construct = row["dba_construct"]
    conv_s    = row["convergence_score"] / 3.0

    comp_raw = composite_means.get(construct, np.nan)
    comp_n   = (comp_raw - comp_min)/(comp_max - comp_min) \
               if not np.isnan(comp_raw) and comp_max > comp_min \
               else 0.5

    sent_raw = sent_by_construct.get(construct, 0.0)
    sent_n   = abs(sent_raw) if not np.isnan(sent_raw) else 0.0

    shap_raw = shap_ranks.get(construct, np.nan)
    shap_n   = 1 - (shap_raw/shap_max) \
               if not np.isnan(shap_raw) else 0.5

    meta_score = (W_CONV*conv_s + W_COMP*comp_n +
                  W_SENT*sent_n + W_SHAP*shap_n)

    meta_rows.append({
        "dba_construct"      : construct,
        "convergence_score"  : row["convergence_score"],
        "convergence_status" : row["convergence_status"],
        "comp_normalized"    : round(comp_n,3),
        "sent_normalized"    : round(sent_n,3),
        "shap_normalized"    : round(shap_n,3),
        "meta_inference_score": round(meta_score,3),
    })

meta_df = pd.DataFrame(meta_rows).sort_values(
    "meta_inference_score", ascending=False
)

print("\n   Meta-Inference Rankings:")
print(meta_df[["dba_construct","meta_inference_score",
               "convergence_status"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, max(4,len(meta_df)*0.6)))
bar_colors = [status_colors.get(r["convergence_status"],"#888888")
              for _, r in meta_df.iterrows()]
bars = ax.barh(meta_df["dba_construct"],
               meta_df["meta_inference_score"],
               color=bar_colors, alpha=0.85)
ax.set_xlabel("Meta-Inference Score")
ax.set_title("Integrated Evidence Strength by DBA Construct\n"
             "(weighted: convergence 35%, composite 25%, "
             "sentiment 20%, SHAP 20%)",
             fontsize=9)
for bar, score in zip(bars, meta_df["meta_inference_score"]):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f"{score:.3f}", va="center", fontsize=8)
legend_patches = [mpatches.Patch(color=c, label=l)
                  for l,c in status_colors.items()]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "meta_inference_ranking.png", dpi=300)
plt.close()
print("   Saved: meta_inference_ranking.png")

# Meta-inference narrative template
top_construct = meta_df.iloc[0]["dba_construct"]
top_score     = meta_df.iloc[0]["meta_inference_score"]
top_comp      = composite_means.get(top_construct, 0)
top_shap_rank = shap_ranks.get(top_construct, "N/A")
divergent_c   = meta_df[meta_df["convergence_status"]=="Divergent"]

narrative_lines = [
    "=============================================",
    "META-INFERENCE NARRATIVE TEMPLATE",
    "=============================================",
    "",
    f"Construct 1 — {top_construct} (score={top_score:.3f}):",
    "Across qualitative and quantitative streams, "
    f"'{top_construct}' emerges as the most robustly",
    "supported finding. Interview and focus group data indicate",
    "[QUALITATIVE FINDING — fill from Block 1 BERTopic output].",
    f"This is corroborated by a composite score of {top_comp:.2f}",
    f"and a SHAP rank of {top_shap_rank}, confirming that",
    f"'{top_construct}' is the primary driver of GenAI readiness",
    "in Lebanese AMCs. The convergence of NLP-derived themes",
    "with Likert-scale measurement suggests this finding is",
    "not method-dependent.",
    "",
]

for _, row in meta_df.iloc[1:].iterrows():
    c     = row["dba_construct"]
    sc    = row["meta_inference_score"]
    cm    = composite_means.get(c, 0)
    narrative_lines += [
        f"Construct — {c} (score={sc:.3f}):",
        f"Composite score: {cm:.2f}. Convergence: {row['convergence_status']}.",
        "[META-INFERENCE — fill after reviewing joint display].",
        "",
    ]

if not divergent_c.empty:
    narrative_lines += ["Divergent Constructs:", ""]
    for _, row in divergent_c.iterrows():
        c = row["dba_construct"]
        narrative_lines += [
            f"Notably, '{c}' showed divergence between",
            "qualitative and quantitative streams. While participants",
            "expressed [QUALITATIVE SENTIMENT], composite scores",
            "indicated [QUANTITATIVE PATTERN]. This divergence may",
            "reflect [INTERPRETATION] and warrants further investigation.",
            "",
        ]

narrative_lines += ["============================================="]
narrative_text = "\n".join(narrative_lines)
print("\n" + narrative_text)

with open(OUTPUT_DIR + "meta_inference_narratives_template.txt", "w") as f:
    f.write(narrative_text)
print("\n   Saved: meta_inference_narratives_template.txt")


# ─────────────────────────────────────────────────────────────
# STEP 8 — ZIP OUTPUT PACKAGE
# ─────────────────────────────────────────────────────────────
print("\n── STEP 8: OUTPUT PACKAGE ──")

all_output_files = [
    "master_integrated_dataset.csv",
    "convergence_matrix_full.csv",
    "joint_display_template.csv",
    "cluster_theme_profiles.csv",
    "role_integration_profiles.csv",
    "lebanon_context_analysis.csv",
    "meta_inference_narratives_template.txt",
    "convergence_matrix.png",
    "joint_display_table.png",
    "cluster_theme_heatmap.png",
    "cluster_sentiment_barplot.png",
    "cluster_shap_profile.png",
    "role_integration_radar.png",
    "role_sentiment_topic_matrix.png",
    "role_cluster_sankey.png",
    "lebanon_context_network.png",
    "meta_inference_ranking.png",
]

zip_path = OUTPUT_DIR + "thesis_block3_outputs.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for fname in all_output_files:
        fpath = OUTPUT_DIR + fname
        if os.path.exists(fpath):
            zf.write(fpath, fname)

print(f"   Saved: thesis_block3_outputs.zip")
saved_b3 = [f for f in all_output_files
            if os.path.exists(OUTPUT_DIR + f)]
print(f"   Files in package: {len(saved_b3)} of {len(all_output_files)}")


# ─────────────────────────────────────────────────────────────
# STEP 9 — TRUSTWORTHINESS CELL
# ─────────────────────────────────────────────────────────────
print("\n── STEP 9: TRUSTWORTHINESS FRAMEWORK ──")
trust_text = """
TRUSTWORTHINESS MEASURES APPLIED
=================================
Credibility (Internal Validity — Qualitative):
  - Triangulation: three data sources converged via convergence matrix
  - BERTopic + manual coding convergence documented
  - Negative case analysis: divergent constructs explicitly reported
  - Member checking: [TO BE CONDUCTED — placeholder]

Transferability (External Validity — Qualitative):
  - Thick description of Lebanese AMC context preserved
  - Purposive sampling of three leadership roles documented
  - Lebanon-specific terms flagged throughout pipeline

Dependability (Reliability — Qualitative):
  - Full pipeline available as supplementary Colab notebook
  - All analytical steps documented with inline annotation
  - No subjective step undocumented

Confirmability (Objectivity — Qualitative):
  - Meta-inference scores computed algorithmically (weights documented)
  - Audit trail through all intermediate CSV outputs
  - Reflexivity note to be added in Chapter 3

Internal Validity (Quantitative):
  - EFA confirms construct structure prior to modelling
  - Cronbach Alpha validates scale reliability per domain
  - 5-fold cross-validation applied to RF model

Reliability (Quantitative):
  - random_state=42 globally — results fully reproducible
  - CV reports mean ± SD

Integration Rigour (Mixed Methods):
  - Convergent design per Creswell & Plano Clark (2018)
  - Joint display follows published MMR standards
  - Meta-inference explicitly labels convergence level per construct
"""
print(trust_text)
with open(OUTPUT_DIR + "trustworthiness_framework.txt", "w") as f:
    f.write(trust_text)
print("   Saved: trustworthiness_framework.txt")


# ─────────────────────────────────────────────────────────────
# STEP 10 — FINAL PIPELINE SUMMARY
# ─────────────────────────────────────────────────────────────
n_confirmed  = len(conv_df[conv_df["convergence_status"]=="Confirmed"])
n_partial    = len(conv_df[conv_df["convergence_status"]=="Partial"])
n_divergent  = len(conv_df[conv_df["convergence_status"]=="Divergent"])
all_saved    = [f for f in os.listdir(OUTPUT_DIR)
                if f.endswith(".csv") or f.endswith(".png")
                or f.endswith(".pkl") or f.endswith(".txt")]

print("\n" + "="*55)
print("  DBA THESIS — FULL PIPELINE COMPLETE")
print("="*55)
print("  BLOCK 1 — Qualitative ML Analysis")
if os.path.exists(OUTPUT_DIR + "bertopic_topic_overview.csv"):
    b1_topics = pd.read_csv(OUTPUT_DIR + "bertopic_topic_overview.csv")
    n_topics  = len(b1_topics[b1_topics["topic_id"] != -1])
    print(f"    BERTopic topics found   : {n_topics}")
if os.path.exists(OUTPUT_DIR + "bertopic_construct_mapping.csv"):
    b1_map = pd.read_csv(OUTPUT_DIR + "bertopic_construct_mapping.csv")
    print(f"    DBA constructs mapped   : {b1_map['primary_construct'].nunique()}")
print(f"    Arabic segments flagged : {master_df.get('arabic_flag', pd.Series([0])).sum()}")

print("\n  BLOCK 2 — Quantitative Analysis")
if not alpha_df.empty:
    n_valid = len(alpha_df[alpha_df["alpha"] >= 0.70]) \
        if "alpha" in alpha_df.columns else "N/A"
    print(f"    Scales validated        : {n_valid}")
if os.path.exists(OUTPUT_DIR + "clustered_participants.csv"):
    cl_df = pd.read_csv(OUTPUT_DIR + "clustered_participants.csv")
    n_cl  = cl_df["cluster_name"].nunique() \
        if "cluster_name" in cl_df.columns else "N/A"
    cl_names = cl_df["cluster_name"].unique().tolist() \
        if "cluster_name" in cl_df.columns else []
    print(f"    Participant clusters    : {n_cl}")
    print(f"    Cluster names          : {cl_names}")

print("\n  BLOCK 3 — Mixed Methods Integration")
print(f"    Confirmed constructs    : {n_confirmed}")
print(f"    Partial constructs      : {n_partial}")
print(f"    Divergent constructs    : {n_divergent}")
print(f"    Meta-inference top rank : {top_construct}")

print(f"\n  TOTAL OUTPUT FILES      : {len(all_saved)}")
print("-"*55)
print("  CHAPTER MAPPING:")
print("    Chapter 3 (Methodology) <- All block pipelines")
print("    Chapter 4 (Findings)    <- All figures + tables")
print("    Chapter 5 (Discussion)  <- Meta-inference narratives")
print("    Appendices              <- Full Colab notebooks")
print("="*55)
print("\n  Download thesis_block3_outputs.zip from:")
print(f"  {zip_path}")
print("="*55)

✅ Block 3 starting — Mixed Methods Integration

── PREREQUISITE CHECK ──
   ✅  bertopic_document_topics.csv
   ✅  bertopic_construct_mapping.csv
   ✅  sentiment_by_role_topic.csv
   ✅  cooccurrence_edges.csv
   ✅  manual_vs_ml_comparison_template.csv
   ✅  composite_scores.csv
   ✅  clustered_participants.csv
   ✅  cluster_profiles.csv
   ✅  shap_values.csv
   ✅  cronbach_alpha_results.csv
   ⚠️  inferential_results.csv empty or unreadable — using empty DataFrame

   Score columns detected: ['DOMAIN_1_score', 'DOMAIN_2_score', 'DOMAIN_3_score', 'DOMAIN_4_score', 'DOMAIN_5_score']

── STEP 1: MASTER DATASET ──
   Master dataset shape: (86, 55)
   Columns: ['Timestamp', "How important is GenAI for maintaining your medical center's competitive edge?Ê", 'Q1', 'How likely is it that GenAI will lead to significant innovations in patient care?Ê', 'What percentage of your strategic planning involves the use of GenAI?', "Rate the impact of GenAI on your medical center's market positioningÊ", 'H

In [9]:
from google.colab import files
import os

to_download = [
    "quantitative_item_labels.csv",
    "composite_scores.csv",
    "cronbach_alpha_results.csv",
    "bertopic_topic_overview.csv",
    "bertopic_construct_mapping.csv",
    "cluster_profiles.csv",
    "convergence_matrix_full.csv",
    "meta_inference_narratives_template.txt",
    "sentiment_by_role_topic.csv",
    "shap_values.csv",
    "efa_loadings.csv",
    "inferential_results.csv",
]

for f in to_download:
    path = "/content/thesis_outputs/" + f
    if os.path.exists(path):
        files.download(path)
        print(f"✅ {f}")
    else:
        print(f"⚠️  {f} not found")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ quantitative_item_labels.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✅ efa_loadings.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ inferential_results.csv


In [1]:
from google.colab import files
import os

figures_needed = [
    "bertopic_topic_distribution.png",
    "bertopic_construct_heatmap.png",
    "sentiment_heatmap_role_topic.png",
    "efa_scree_plot.png",
    "efa_loading_heatmap.png",
    "shap_summary_beeswarm.png",
    "kmeans_radar_chart.png",
    "meta_inference_ranking.png",
]

for f in figures_needed:
    path = "/content/thesis_outputs/" + f
    if os.path.exists(path):
        files.download(path)
        print(f"✅ {f}")
    else:
        print(f"❌ Missing: {f} — re-run the relevant block")

❌ Missing: bertopic_topic_distribution.png — re-run the relevant block
❌ Missing: bertopic_construct_heatmap.png — re-run the relevant block
❌ Missing: sentiment_heatmap_role_topic.png — re-run the relevant block
❌ Missing: efa_scree_plot.png — re-run the relevant block
❌ Missing: efa_loading_heatmap.png — re-run the relevant block
❌ Missing: shap_summary_beeswarm.png — re-run the relevant block
❌ Missing: kmeans_radar_chart.png — re-run the relevant block
❌ Missing: meta_inference_ranking.png — re-run the relevant block


In [ ]:
from google.colab import drive
drive.mount('/content/drive')